In [1]:
!pip install optuna fairlearn folktables ucimlrepo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 3.9 MB/s eta 0:00:00


In [2]:
# import os, gc, warnings, json, time, copy, random
# os.environ.setdefault("PYTHONHASHSEED", "42")
# os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

# import numpy as np
# import pandas as pd
# import joblib
# import torch
# import torch.nn as nn
# import torch.optim as optim
# from torch.utils.data import DataLoader, TensorDataset
# from tqdm.auto import tqdm
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import accuracy_score, roc_auc_score

# warnings.filterwarnings("ignore")
# torch.use_deterministic_algorithms(True, warn_only=False)
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark     = False

# try:
#     from folktables import ACSDataSource, ACSEmployment
#     HAS_FOLKTABLES = True
# except ImportError:
#     HAS_FOLKTABLES = False

# try:
#     from fairlearn.adversarial import AdversarialFairnessClassifier
#     HAS_FAIRLEARN = True
# except ImportError:
#     HAS_FAIRLEARN = False

# SEED          = 42
# N_SEEDS       = 3
# SEEDS         = list(range(SEED, SEED + N_SEEDS))
# DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# TARGET_EO     = 0.05
# TARGET_DP     = 0.10
# CACHE_VERSION = "v17"          # bumped so stale v16 caches are never silently reused

# ACS_STATES          = ["CA"]
# TOP_K_INTERSECTIONS = 5
# MAX_SG_ATTRS        = 2
# PLATEAU_PATIENCE    = 2
# PARETO_LAMBDA       = 0.10
# EO_GAP_FIRE_THRESH  = 0.015
# DP_COEF_FRAC        = 0.40

# RUN_DATASETS = ["acs_employment", "adult", "communities_crime"]

# DATASET_CONFIG = {
#     "acs_employment": {
#         "sens_attrs":      ["race_White", "race_Black", "race_Other", "sex_Female"],
#         "reweight_labels": True,
#         "min_frac": 0.02, "max_frac": 0.40,
#     },
#     "adult": {
#         "sens_attrs":      ["race_Black", "race_White", "sex_Female"],
#         "reweight_labels": True,
#         "min_frac": 0.02, "max_frac": 0.40,
#     },
#     "communities_crime": {
#         "sens_attrs":      ["high_black_pct", "low_income", "high_unemployment"],
#         "reweight_labels": True,
#         "min_frac": 0.05, "max_frac": 0.45,
#     },
# }

# BASE_PARAMS = {
#     "acs_employment": {
#         "hidden_dim": 96, "dropout": 0.15, "lr": 5e-4,
#         "batch": 4096, "epochs": 55,
#         "fairness_coef": 28.0,
#         "eval_every": 4, "patience": 16,
#         "min_acc_floor": 0.68,
#         "adv_lr_mult": 4.0, "adv_steps": 3,
#         "pre_epochs": 8,
#         "warmup_adv_epochs": 8, "task_lw": 2.5, "grl_max": 0.65,
#     },
#     "adult": {
#         "hidden_dim": 96, "dropout": 0.15, "lr": 5e-4,
#         "batch": 2048, "epochs": 65,
#         "fairness_coef": 45.0,
#         "eval_every": 4, "patience": 18,
#         "min_acc_floor": 0.72,
#         "adv_lr_mult": 5.0, "adv_steps": 4,
#         "pre_epochs": 10,
#         "warmup_adv_epochs": 8, "task_lw": 2.0, "grl_max": 0.80,
#     },
#     "communities_crime": {
#         "hidden_dim": 64, "dropout": 0.20, "lr": 3e-4,
#         "batch": 256, "epochs": 140,
#         "fairness_coef": 130.0,
#         "eval_every": 4, "patience": 32,
#         "min_acc_floor": 0.50,
#         "adv_lr_mult": 9.0, "adv_steps": 8,
#         "pre_epochs": 12,
#         "warmup_adv_epochs": 6, "task_lw": 1.0, "grl_max": 0.95,
#     },
# }

# METHOD_LABELS = {
#     "vanilla_nn":        "Vanilla NN",
#     "zhang_adversarial": "Zhang / No-Auditor",
#     "fairlearn_adv":     "FairLearn Adv",
#     "agad_full":         "AGAD (full)",
# }
# METHOD_COLORS = {
#     "vanilla_nn":        "#e74c3c",
#     "zhang_adversarial": "#f39c12",
#     "fairlearn_adv":     "#2ecc71",
#     "agad_full":         "#3498db",
# }

# METHODS = ["vanilla_nn", "zhang_adversarial"]
# if HAS_FAIRLEARN:
#     METHODS.append("fairlearn_adv")
# METHODS.append("agad_full")


# # ──────────────────────────────────────────────────────────────────────────────
# # Utilities
# # ──────────────────────────────────────────────────────────────────────────────

# def set_seed(s=SEED):
#     """Seed Python, NumPy, and PyTorch for reproducibility."""
#     random.seed(s)
#     np.random.seed(s)
#     torch.manual_seed(s)
#     torch.cuda.manual_seed_all(s)

# def to_dense(X):
#     return X.toarray() if hasattr(X, "toarray") else np.array(X)

# def ensure_binary(sv):
#     sv = np.array(sv).flatten()
#     u  = np.unique(sv[~np.isnan(sv.astype(float))])
#     if len(u) <= 1: return np.zeros(len(sv), dtype=int)
#     if len(u) == 2: return (sv == u[1]).astype(int)
#     maj = u[np.argmax([(sv == v).sum() for v in u])]
#     return (sv != maj).astype(int)

# def stratified_split(X, y, sens_list, test_size=0.2):
#     key = np.array(y).astype(str).copy()
#     for s in sens_list:
#         key = key + "_" + np.array(s).astype(str)
#     u, c = np.unique(key, return_counts=True)
#     key[np.isin(key, u[c < 2])] = np.array(y)[np.isin(key, u[c < 2])].astype(str)
#     try:
#         return train_test_split(np.arange(len(y)), test_size=test_size,
#                                 stratify=key, random_state=SEED)
#     except Exception:
#         return train_test_split(np.arange(len(y)), test_size=test_size,
#                                 stratify=y, random_state=SEED)

# def build_val_split(y_train, sv_list, frac=0.15, seed=SEED):
#     key = np.array(y_train).astype(str)
#     for sv in sv_list:
#         key = key + "_" + ensure_binary(np.array(sv)).astype(str)
#     u, c = np.unique(key, return_counts=True)
#     key[np.isin(key, u[c < 2])] = np.array(y_train)[np.isin(key, u[c < 2])].astype(str)
#     try:
#         return train_test_split(np.arange(len(y_train)), test_size=frac,
#                                 stratify=key, random_state=seed)
#     except Exception:
#         return train_test_split(np.arange(len(y_train)), test_size=frac,
#                                 stratify=y_train, random_state=seed)

# def safe_auc(y_true, y_score):
#     try:
#         if len(np.unique(y_true)) < 2: return float("nan")
#         return roc_auc_score(y_true, y_score)
#     except Exception:
#         return float("nan")

# def _to_float(v):
#     try:
#         f = float(v)
#         return None if (f != f) else f
#     except Exception:
#         return None

# def _banner(title, width=80, char="═"):
#     pad = max(0, width - len(title) - 4)
#     return f"\n  {char*2} {title} {char * pad}"


# # ──────────────────────────────────────────────────────────────────────────────
# # Label reweighting
# # ──────────────────────────────────────────────────────────────────────────────

# def compute_label_reweights(y, sv_list):
#     y    = np.array(y)
#     n    = len(y)
#     p_y1 = np.clip(y.mean(), 1e-3, 1-1e-3)
#     p_y0 = 1.0 - p_y1
#     w_sum = np.zeros(n, dtype=np.float64)
#     count = 0
#     for sv in sv_list:
#         sv_b = ensure_binary(sv)
#         if len(np.unique(sv_b)) < 2: continue
#         w = np.ones(n, dtype=np.float64)
#         for g in [0, 1]:
#             mask = sv_b == g
#             if mask.sum() == 0: continue
#             p_y1_g = np.clip(y[mask].mean(), 1e-3, 1-1e-3)
#             w[mask & (y==1)] = p_y1 / p_y1_g
#             w[mask & (y==0)] = p_y0 / (1 - p_y1_g)
#         w_sum += w
#         count += 1
#     if count == 0: return np.ones(n, dtype=np.float32)
#     w_avg = w_sum / count
#     return (w_avg / w_avg.mean()).astype(np.float32)


# # ──────────────────────────────────────────────────────────────────────────────
# # Fairness metrics
# # ──────────────────────────────────────────────────────────────────────────────

# def compute_eo_binary(y_true, y_prob, sv):
#     sv = ensure_binary(sv)
#     if len(np.unique(sv)) != 2: return 0.0
#     tprs, fprs = [], []
#     for g in [0, 1]:
#         pos = (sv==g) & (y_true==1)
#         neg = (sv==g) & (y_true==0)
#         tprs.append(float(y_prob[pos].mean()) if pos.sum() > 0 else 0.0)
#         fprs.append(float(y_prob[neg].mean()) if neg.sum() > 0 else 0.0)
#     return max(abs(tprs[0]-tprs[1]), abs(fprs[0]-fprs[1]))

# def compute_eo_multi(y_true, y_prob, sv_list, attr_names):
#     per = [compute_eo_binary(y_true, y_prob, sv) for sv in sv_list]
#     return per, max(per) if per else 0.0

# def compute_dp(y_prob, sv):
#     sv = ensure_binary(sv)
#     if len(np.unique(sv)) != 2: return 0.0
#     r0 = float(y_prob[sv==0].mean()) if (sv==0).sum() > 0 else 0.5
#     r1 = float(y_prob[sv==1].mean()) if (sv==1).sum() > 0 else 0.5
#     return abs(r0 - r1)

# def compute_dp_multi(y_prob, sv_list, attr_names):
#     per = [compute_dp(y_prob, sv) for sv in sv_list]
#     return per, max(per) if per else 0.0

# def compute_ece(y_true, y_prob, n_bins=15):
#     bins = np.linspace(0, 1, n_bins+1)
#     ece  = 0.0
#     for i in range(n_bins):
#         mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
#         if mask.sum() == 0: continue
#         ece += mask.sum() * abs(y_true[mask].mean() - y_prob[mask].mean())
#     return float(ece / len(y_true))

# def _soft_eo(prob, sv_b, yf, eps):
#     sf = sv_b.float()
#     g0y1 = (1-sf)*yf;     g1y1 = sf*yf
#     g0y0 = (1-sf)*(1-yf); g1y0 = sf*(1-yf)
#     if any(gg.sum() < 1e-6 for gg in [g0y1, g1y1, g0y0, g1y0]):
#         return torch.tensor(0., device=prob.device)
#     tpr0 = (prob*g0y1).sum() / (g0y1.sum()+eps)
#     tpr1 = (prob*g1y1).sum() / (g1y1.sum()+eps)
#     fpr0 = (prob*g0y0).sum() / (g0y0.sum()+eps)
#     fpr1 = (prob*g1y0).sum() / (g1y0.sum()+eps)
#     return torch.max(torch.abs(tpr0-tpr1), torch.abs(fpr0-fpr1))

# def _soft_dp(prob, sv_b, eps):
#     sf  = sv_b.float()
#     n0  = (1-sf).sum() + eps
#     n1  = sf.sum()     + eps
#     r0  = (prob * (1-sf)).sum() / n0
#     r1  = (prob *    sf ).sum() / n1
#     return torch.abs(r0 - r1)


# # ──────────────────────────────────────────────────────────────────────────────
# # Subgroup enumeration
# # ──────────────────────────────────────────────────────────────────────────────

# def _build_disjoint_subgroups(y_true, y_prob, sv_list, attr_names,
#                                min_frac, max_frac):
#     n      = len(y_true)
#     min_sz = max(10, int(min_frac * n))
#     max_sz = int(max_frac * n)
#     valid_svs, valid_names = [], []
#     for sv, nm in zip(sv_list, attr_names):
#         sv_a = np.array(sv).astype(int)
#         if len(np.unique(sv_a)) < 2: continue
#         valid_svs.append(sv_a); valid_names.append(nm)
#     n_attr = len(valid_svs)
#     if n_attr == 0: return []
#     subgroups = []; seen = set()
#     for mask in range(1, 2**n_attr):
#         active = [i for i in range(n_attr) if mask & (1 << i)]
#         if len(active) > MAX_SG_ATTRS: continue
#         for vals in range(2**len(active)):
#             req = [(vals >> j) & 1 for j in range(len(active))]
#             mem = np.ones(n, dtype=bool)
#             for ai, rv in zip(active, req): mem &= (valid_svs[ai] == rv)
#             key = mem.tobytes()
#             if key in seen: continue
#             seen.add(key); sz = mem.sum()
#             if sz < min_sz or sz > max_sz: continue
#             if len(np.unique(y_true[mem])) < 2: continue
#             subgroups.append((mem, [valid_names[i] for i in active], req,
#                               [(valid_names[i], req[j]) for j, i in enumerate(active)]))
#     return subgroups

# def compute_eo_subgroups_pairwise(y_true, y_prob, sv_list, attr_names,
#                                    min_frac, max_frac):
#     subgroups = _build_disjoint_subgroups(
#         y_true, y_prob, sv_list, attr_names, min_frac, max_frac)
#     n = len(y_true)
#     if len(subgroups) < 2: return 0., None, np.zeros(n, dtype=int)
#     worst = 0.; worst_grp = None; worst_mem = np.zeros(n, dtype=int)
#     for i in range(len(subgroups)):
#         memi, namei, reqi, _ = subgroups[i]
#         for j in range(i+1, len(subgroups)):
#             memj, _, _, _ = subgroups[j]
#             if int((memi & memj).sum()) > 0: continue
#             yi = y_true[memi]; pi = y_prob[memi]
#             yj = y_true[memj]; pj = y_prob[memj]
#             tpr_i = float(pi[yi==1].mean()) if (yi==1).sum() > 0 else 0.
#             tpr_j = float(pj[yj==1].mean()) if (yj==1).sum() > 0 else 0.
#             fpr_i = float(pi[yi==0].mean()) if (yi==0).sum() > 0 else 0.
#             fpr_j = float(pj[yj==0].mean()) if (yj==0).sum() > 0 else 0.
#             eo = max(abs(tpr_i-tpr_j), abs(fpr_i-fpr_j))
#             if eo > worst:
#                 worst = eo
#                 worst_grp = {"attrs": namei, "vals": reqi, "n": int(memi.sum())}
#                 worst_mem = memi.astype(int)
#     return float(worst), worst_grp, worst_mem

# def compute_eo_subgroups(y_true, y_prob, sv_list, attr_names,
#                           min_frac, max_frac):
#     n      = len(y_true)
#     min_sz = max(10, int(min_frac * n))
#     max_sz = int(max_frac * n)
#     valid_svs, valid_names = [], []
#     for sv, nm in zip(sv_list, attr_names):
#         sv_a = np.array(sv).astype(int)
#         if len(np.unique(sv_a)) < 2: continue
#         valid_svs.append(sv_a); valid_names.append(nm)
#     n_attr = len(valid_svs)
#     if n_attr == 0: return 0., None, np.zeros(n, dtype=int)
#     worst = 0.; worst_grp = None; worst_mem = np.zeros(n, dtype=int)
#     seen = set()
#     for mask in range(1, 2**n_attr):
#         active = [i for i in range(n_attr) if mask & (1 << i)]
#         if len(active) > MAX_SG_ATTRS: continue
#         for vals in range(2**len(active)):
#             req = [(vals >> j) & 1 for j in range(len(active))]
#             mem = np.ones(n, dtype=bool)
#             for ai, rv in zip(active, req): mem &= (valid_svs[ai] == rv)
#             key = mem.tobytes()
#             if key in seen: continue
#             seen.add(key); sz = mem.sum()
#             if sz < min_sz or sz > max_sz: continue
#             sg_p = y_prob[mem];  sg_t = y_true[mem]
#             ot_p = y_prob[~mem]; ot_t = y_true[~mem]
#             if len(np.unique(sg_t)) < 2 or len(np.unique(ot_t)) < 2: continue
#             tpr_sg  = float(sg_p[sg_t==1].mean()) if (sg_t==1).sum() > 0 else 0.
#             tpr_oth = float(ot_p[ot_t==1].mean()) if (ot_t==1).sum() > 0 else 0.
#             fpr_sg  = float(sg_p[sg_t==0].mean()) if (sg_t==0).sum() > 0 else 0.
#             fpr_oth = float(ot_p[ot_t==0].mean()) if (ot_t==0).sum() > 0 else 0.
#             eo = max(abs(tpr_sg-tpr_oth), abs(fpr_sg-fpr_oth))
#             if eo > worst:
#                 worst = eo
#                 worst_grp = {"attrs": [valid_names[i] for i in active],
#                              "vals": req, "n": int(sz)}
#                 worst_mem = mem.astype(int)
#     return float(worst), worst_grp, worst_mem


# def _make_membership(sv_bin_list, attr_names, active_attr_vals, n):
#     mem = np.ones(n, dtype=bool)
#     name_to_sv = {nm: sv for nm, sv in zip(attr_names, sv_bin_list)}
#     for attr_nm, req_val in active_attr_vals:
#         if attr_nm not in name_to_sv:
#             return np.zeros(n, dtype=bool)
#         mem &= (name_to_sv[attr_nm] == req_val)
#     return mem


# def eo_gap_auditor(y_val, p_val, sv_val_bin, attr_names,
#                    sv_tr_bin, min_frac, max_frac):
#     subgroups = _build_disjoint_subgroups(
#         y_val, p_val, sv_val_bin, attr_names, min_frac, max_frac)
#     if len(subgroups) < 2: return []
#     n_tr = len(sv_tr_bin[0])
#     pairs = []
#     for i in range(len(subgroups)):
#         memi, namei, reqi, spec_i = subgroups[i]
#         for j in range(i+1, len(subgroups)):
#             memj, namej, reqj, spec_j = subgroups[j]
#             if int((memi & memj).sum()) > 0: continue
#             yi = y_val[memi]; pi = p_val[memi]
#             yj = y_val[memj]; pj = p_val[memj]
#             tpr_i = float(pi[yi==1].mean()) if (yi==1).sum() > 0 else 0.
#             tpr_j = float(pj[yj==1].mean()) if (yj==1).sum() > 0 else 0.
#             fpr_i = float(pi[yi==0].mean()) if (yi==0).sum() > 0 else 0.
#             fpr_j = float(pj[yj==0].mean()) if (yj==0).sum() > 0 else 0.
#             eo = max(abs(tpr_i-tpr_j), abs(fpr_i-fpr_j))
#             if eo >= EO_GAP_FIRE_THRESH:
#                 mem_i_tr = _make_membership(sv_tr_bin, attr_names, spec_i, n_tr)
#                 mem_j_tr = _make_membership(sv_tr_bin, attr_names, spec_j, n_tr)
#                 pairs.append((eo,
#                               mem_i_tr.astype(np.float32),
#                               mem_j_tr.astype(np.float32),
#                               namei, namej))
#     if not pairs: return []
#     pairs.sort(key=lambda x: -x[0])
#     flagged = []
#     for eo, memi_tr, memj_tr, namei, namej in pairs[:TOP_K_INTERSECTIONS]:
#         name_i_str = "+".join(namei) if namei else "?"
#         name_j_str = "+".join(namej) if isinstance(namej, list) else str(namej)
#         tqdm.write(f"    │   EO-gap={eo:.4f}  [{name_i_str}] vs [{name_j_str}]")
#         flagged.append((memi_tr, memj_tr, eo, namei, namej))
#     return flagged


# # ──────────────────────────────────────────────────────────────────────────────
# # Model architecture
# # ──────────────────────────────────────────────────────────────────────────────

# class GradientReversalFn(torch.autograd.Function):
#     @staticmethod
#     def forward(ctx, x, alpha):
#         ctx.alpha = alpha; return x.clone()
#     @staticmethod
#     def backward(ctx, grad):
#         return -ctx.alpha * grad, None

# class GRL(nn.Module):
#     def __init__(self, alpha=1.0):
#         super().__init__(); self.alpha = alpha
#     def forward(self, x):
#         return GradientReversalFn.apply(x, self.alpha)

# class Encoder(nn.Module):
#     def __init__(self, in_dim, hidden_dim, dropout):
#         super().__init__()
#         h = hidden_dim; self.rep_dim = h // 2 + 32
#         self.net = nn.Sequential(
#             nn.Linear(in_dim, h), nn.BatchNorm1d(h),
#             nn.GELU(), nn.Dropout(dropout),
#             nn.Linear(h, self.rep_dim), nn.BatchNorm1d(self.rep_dim),
#             nn.GELU(), nn.Dropout(dropout*0.5))
#         for m in self.modules():
#             if isinstance(m, nn.Linear):
#                 nn.init.kaiming_normal_(m.weight, nonlinearity="linear")
#                 if m.bias is not None: nn.init.zeros_(m.bias)
#     def forward(self, x): return self.net(x)

# class TaskHead(nn.Module):
#     def __init__(self, rep_dim):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(rep_dim, rep_dim//2), nn.GELU(),
#             nn.Linear(rep_dim//2, 1))
#         for m in self.modules():
#             if isinstance(m, nn.Linear):
#                 nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
#     def forward(self, rep): return self.net(rep).squeeze(1)

# class AdversaryHead(nn.Module):
#     def __init__(self, rep_dim, hidden_dim, dropout):
#         super().__init__()
#         self.grl = GRL(alpha=1.0); h = hidden_dim // 2
#         self.net = nn.Sequential(
#             nn.Linear(rep_dim, h), nn.GELU(), nn.Dropout(dropout),
#             nn.Linear(h, h//2),   nn.GELU(), nn.Dropout(dropout*0.5),
#             nn.Linear(h//2, 1))
#         for m in self.modules():
#             if isinstance(m, nn.Linear):
#                 nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
#     def set_alpha(self, a): self.grl.alpha = a
#     def forward(self, z): return self.net(self.grl(z)).squeeze(1)

# def _make_loader(tensors, batch_size, seed):
#     ds = TensorDataset(*tensors)
#     g  = torch.Generator(); g.manual_seed(seed)
#     return DataLoader(ds, batch_size=batch_size, shuffle=True,
#                       drop_last=True, generator=g, num_workers=0)


# # ──────────────────────────────────────────────────────────────────────────────
# # Pretraining
# # ──────────────────────────────────────────────────────────────────────────────

# def pretrain_encoder(X_tr, y_tr, wt, params, seed):
#     set_seed(seed); p = params
#     def _t(a): return torch.tensor(np.array(a), dtype=torch.float32).to(DEVICE)
#     Xt = _t(X_tr); yt = _t(y_tr); wt_t = _t(wt)
#     enc = Encoder(X_tr.shape[1], p["hidden_dim"], p["dropout"]).to(DEVICE)
#     th  = TaskHead(enc.rep_dim).to(DEVICE)
#     bce = nn.BCEWithLogitsLoss(reduction="none")
#     opt = optim.AdamW(list(enc.parameters()) + list(th.parameters()),
#                       lr=p["lr"], weight_decay=1e-4)
#     loader = _make_loader([Xt, yt, wt_t], p["batch"], seed)
#     enc.train(); th.train()
#     for ep in range(p.get("pre_epochs", 8)):
#         for xb, yb, wb in loader:
#             opt.zero_grad(set_to_none=True)
#             (bce(th(enc(xb)), yb) * wb).mean().backward()
#             opt.step()
#     tqdm.write(f"    │ pretrain: {p.get('pre_epochs', 8)} epochs done")
#     return copy.deepcopy(enc.state_dict()), copy.deepcopy(th.state_dict()), enc.rep_dim


# # ──────────────────────────────────────────────────────────────────────────────
# # Evaluation helper
# # ──────────────────────────────────────────────────────────────────────────────

# def _eval_and_log(proba, y_test, sv_te_bin, attr_names,
#                   min_frac, max_frac, tag="method"):
#     pred          = (proba > 0.5).astype(int)
#     per_eo, wc_eo = compute_eo_multi(y_test, proba, sv_te_bin, attr_names)
#     per_dp, wc_dp = compute_dp_multi(proba, sv_te_bin, attr_names)
#     sg_w, wg, _   = compute_eo_subgroups_pairwise(
#         y_test, proba, sv_te_bin, attr_names, min_frac, max_frac)
#     ece = compute_ece(y_test, proba)
#     acc = float(accuracy_score(y_test, pred))
#     auc = float(safe_auc(y_test, proba))
#     eo_s = "✓" if sg_w < TARGET_EO else "✗"
#     dp_s = "✓" if wc_dp < TARGET_DP else "✗"
#     tqdm.write(f"    │ [{tag}]  WC-EO={wc_eo:.4f}  SG-EO={sg_w:.4f}{eo_s}"
#                f"  WC-DP={wc_dp:.4f}{dp_s}  Acc={acc:.4f}  AUC={auc:.4f}"
#                f"  ECE={ece:.4f}")
#     if wg:
#         attrs_str = "+".join(wg.get("attrs", []))
#         tqdm.write(f"    │         worst subgroup: [{attrs_str}]  n={wg.get('n','?')}")
#     return {"eo": float(wc_eo), "sg_eo": float(sg_w), "acc": acc, "auc": auc,
#             "dp": float(wc_dp), "ece": ece,
#             "per_attr_eo": [float(x) for x in per_eo],
#             "per_attr_dp": [float(x) for x in per_dp],
#             "subgroup_worst": float(sg_w), "n_inter_heads": 0}


# # ──────────────────────────────────────────────────────────────────────────────
# # Baseline methods
# # ──────────────────────────────────────────────────────────────────────────────

# def run_vanilla_nn(X_tr, y_tr, wt, X_val, y_val, X_test, y_test,
#                    sv_te_list, sv_val_list, attr_names,
#                    enc_sd, th_sd, rep_dim, params, ds_cfg, seed):
#     set_seed(seed); p = params
#     min_frac = ds_cfg["min_frac"]; max_frac = ds_cfg["max_frac"]
#     def _t(a): return torch.tensor(np.array(a), dtype=torch.float32).to(DEVICE)
#     Xt = _t(X_tr); yt = _t(y_tr); wt_t = _t(wt)
#     Xv = _t(X_val); Xte = _t(X_test)
#     sv_te_bin  = [ensure_binary(sv) for sv in sv_te_list]
#     sv_val_bin = [ensure_binary(sv) for sv in sv_val_list]
#     enc = Encoder(X_tr.shape[1], p["hidden_dim"], p["dropout"]).to(DEVICE)
#     th  = TaskHead(rep_dim).to(DEVICE)
#     enc.load_state_dict(enc_sd); th.load_state_dict(th_sd)
#     bce    = nn.BCEWithLogitsLoss(reduction="none")
#     opt    = optim.AdamW(list(enc.parameters()) + list(th.parameters()),
#                          lr=p["lr"], weight_decay=1e-4)
#     sched  = optim.lr_scheduler.CosineAnnealingLR(opt, p["epochs"], eta_min=p["lr"]*0.01)
#     loader = _make_loader([Xt, yt, wt_t], p["batch"], seed)
#     best_score = np.inf; best_enc = copy.deepcopy(enc.state_dict())
#     best_th    = copy.deepcopy(th.state_dict()); no_imp = 0
#     for epoch in range(p["epochs"]):
#         enc.train(); th.train()
#         for xb, yb, wb in loader:
#             opt.zero_grad(set_to_none=True)
#             (bce(th(enc(xb)), yb) * wb).mean().backward(); opt.step()
#         sched.step()
#         if epoch % p["eval_every"] == 0:
#             enc.eval(); th.eval()
#             with torch.no_grad():
#                 pv = torch.sigmoid(th(enc(Xv))).cpu().numpy()
#             val_sg, _, _ = compute_eo_subgroups(
#                 y_val, pv, sv_val_bin, attr_names, min_frac, max_frac)
#             val_auc = safe_auc(y_val, pv)
#             score = val_sg + PARETO_LAMBDA*(1-val_auc) if not np.isnan(val_auc) else val_sg
#             if score < best_score:
#                 best_score = score; best_enc = copy.deepcopy(enc.state_dict())
#                 best_th = copy.deepcopy(th.state_dict()); no_imp = 0
#             else:
#                 no_imp += p["eval_every"]
#             if no_imp >= p["patience"] * p["eval_every"]: break
#     enc.load_state_dict(best_enc); th.load_state_dict(best_th)
#     enc.eval(); th.eval()
#     with torch.no_grad():
#         proba = torch.sigmoid(th(enc(Xte))).cpu().numpy()
#     return _eval_and_log(proba, y_test, sv_te_bin, attr_names,
#                          min_frac, max_frac, tag="vanilla-nn")


# def run_zhang_adversarial(X_tr, y_tr, wt, sv_tr_list, sv_val_list, attr_names,
#                           X_val, y_val, X_test, y_test, sv_te_list,
#                           enc_sd, th_sd, rep_dim, params, ds_cfg, seed):
#     set_seed(seed); p = params
#     min_frac = ds_cfg["min_frac"]; max_frac = ds_cfg["max_frac"]
#     def _t(a): return torch.tensor(np.array(a), dtype=torch.float32).to(DEVICE)
#     sv_tr_bin  = [ensure_binary(sv) for sv in sv_tr_list]
#     sv_te_bin  = [ensure_binary(sv) for sv in sv_te_list]
#     sv_val_bin = [ensure_binary(sv) for sv in sv_val_list]
#     n_base     = len(attr_names)
#     Xt  = _t(X_tr); yt = _t(y_tr); wt_t = _t(wt)
#     Xte = _t(X_test); Xv = _t(X_val)
#     st_list = [_t(sv) for sv in sv_tr_bin]
#     enc  = Encoder(X_tr.shape[1], p["hidden_dim"], p["dropout"]).to(DEVICE)
#     th   = TaskHead(rep_dim).to(DEVICE)
#     enc.load_state_dict(enc_sd); th.load_state_dict(th_sd)
#     advs = nn.ModuleList([
#         AdversaryHead(rep_dim, p["hidden_dim"], p["dropout"]).to(DEVICE)
#         for _ in attr_names])
#     bce     = nn.BCEWithLogitsLoss(reduction="none")
#     opt_enc = optim.AdamW(list(enc.parameters()) + list(th.parameters()),
#                           lr=p["lr"], weight_decay=1e-4)
#     opt_adv = optim.AdamW(advs.parameters(),
#                           lr=p["lr"] * p["adv_lr_mult"], weight_decay=1e-4)
#     sched   = optim.lr_scheduler.CosineAnnealingLR(
#         opt_enc, p["epochs"], eta_min=p["lr"]*0.01)
#     loader = _make_loader([Xt, yt, wt_t] + st_list, p["batch"], seed)
#     best_score = np.inf; best_enc = copy.deepcopy(enc.state_dict())
#     best_th_sd = copy.deepcopy(th.state_dict()); no_imp = 0
#     for epoch in range(p["epochs"]):
#         grl_alpha = p["grl_max"] * max(0.1, epoch / max(1, p["epochs"]))
#         for h in advs: h.set_alpha(grl_alpha)
#         enc.train(); th.train(); advs.train()
#         for batch in loader:
#             xb        = batch[0]; yb = batch[1]; wb = batch[2]
#             sv_b_list = [batch[3+i] for i in range(n_base)]
#             z_d    = enc(xb).detach()
#             a_loss = sum(nn.functional.binary_cross_entropy_with_logits(
#                 advs[i](z_d), sv_b_list[i].float()) for i in range(n_base)) / n_base
#             opt_adv.zero_grad(set_to_none=True); a_loss.backward()
#             nn.utils.clip_grad_norm_(advs.parameters(), 1.0); opt_adv.step()
#             opt_enc.zero_grad(set_to_none=True)
#             z = enc(xb)
#             task_loss = (bce(th(z), yb) * wb).mean()
#             grl_loss  = sum(nn.functional.binary_cross_entropy_with_logits(
#                 advs[i](z), sv_b_list[i].float()) for i in range(n_base)) / n_base
#             (p["task_lw"] * task_loss + grl_loss).backward()
#             nn.utils.clip_grad_norm_(list(enc.parameters()) + list(th.parameters()), 0.5)
#             opt_enc.step()
#         sched.step()
#         if epoch % p["eval_every"] == 0:
#             enc.eval(); th.eval()
#             with torch.no_grad():
#                 pv = torch.sigmoid(th(enc(Xv))).cpu().numpy()
#             val_sg, _, _ = compute_eo_subgroups(
#                 y_val, pv, sv_val_bin, attr_names, min_frac, max_frac)
#             val_auc = safe_auc(y_val, pv)
#             val_acc = accuracy_score(y_val, (pv > 0.5).astype(int))
#             score = val_sg + PARETO_LAMBDA*(1-val_auc) if not np.isnan(val_auc) else val_sg
#             if val_acc >= p["min_acc_floor"] and score < best_score:
#                 best_score = score; best_enc = copy.deepcopy(enc.state_dict())
#                 best_th_sd = copy.deepcopy(th.state_dict()); no_imp = 0
#             else:
#                 no_imp += p["eval_every"]
#             if no_imp >= p["patience"] * p["eval_every"]: break
#     enc.load_state_dict(best_enc); th.load_state_dict(best_th_sd)
#     enc.eval(); th.eval()
#     with torch.no_grad():
#         proba = torch.sigmoid(th(enc(Xte))).cpu().numpy()
#     return _eval_and_log(proba, y_test, sv_te_bin, attr_names,
#                          min_frac, max_frac, tag="zhang-adv")


# def _group_encode(sv_bin_list):
#     n = len(sv_bin_list[0]); g = np.zeros(n, dtype=int)
#     for i, sv in enumerate(sv_bin_list):
#         g += sv.astype(int) * (2**i)
#     return g

# def run_fairlearn_adv(X_tr, y_tr, X_test, y_test,
#                       sv_tr_list, sv_te_list, attr_names,
#                       params, ds_cfg, seed):
#     if not HAS_FAIRLEARN:
#         tqdm.write("    │ [fairlearn-adv] not installed — pip install fairlearn")
#         return None
#     set_seed(seed); p = params
#     min_frac = ds_cfg["min_frac"]; max_frac = ds_cfg["max_frac"]
#     sv_tr_bin = [ensure_binary(sv) for sv in sv_tr_list]
#     sv_te_bin = [ensure_binary(sv) for sv in sv_te_list]
#     sf_tr = _group_encode(sv_tr_bin)
#     n_groups = int(np.max(sf_tr)) + 1
#     h = p["hidden_dim"]
#     alpha = max(1.0, p["fairness_coef"] / 15.0)
#     tqdm.write(f"    │ [fairlearn-adv] n_groups={n_groups}  alpha={alpha:.2f}"
#                f"  epochs={p['epochs']}")
#     try:
#         clf = AdversarialFairnessClassifier(
#             predictor_model=[h, "relu", h // 2, "relu"],
#             adversary_model=[h // 4, "relu"],
#             constraints="equalized_odds",
#             learning_rate=p["lr"],
#             alpha=alpha,
#             epochs=p["epochs"],
#             batch_size=p["batch"],
#             shuffle=True,
#             progress_updates=None,
#             random_state=seed,
#         )
#         clf.fit(X_tr, y_tr, sensitive_features=sf_tr)
#     except Exception as e:
#         tqdm.write(f"    │ [fairlearn-adv] ERROR: {e}")
#         return None
#     try:
#         proba = clf.predict_proba(X_test)[:, 1]
#     except Exception:
#         proba = clf.predict(X_test).astype(float)
#     return _eval_and_log(proba, y_test, sv_te_bin, attr_names,
#                          min_frac, max_frac, tag="fairlearn-adv")


# # ──────────────────────────────────────────────────────────────────────────────
# # AGAD
# # ──────────────────────────────────────────────────────────────────────────────

# def _train_agad(X_tr, y_tr, wt, sv_tr_list, attr_names,
#                 X_val, y_val, sv_val_list,
#                 X_test, y_test, sv_te_list,
#                 enc_sd, th_sd, rep_dim,
#                 params, ds_cfg, verbose, seed):
#     p        = params
#     min_frac = ds_cfg["min_frac"]; max_frac = ds_cfg["max_frac"]
#     grl_max  = p["grl_max"]; fc = p["fairness_coef"]
#     dp_fc    = fc * DP_COEF_FRAC
#     task_lw  = p["task_lw"]; min_acc = p["min_acc_floor"]
#     n_base   = len(sv_tr_list)
#     set_seed(seed)
#     eps = torch.tensor(1e-4, device=DEVICE)

#     def _t(a, dtype=torch.float32):
#         return torch.tensor(np.array(a), dtype=dtype).to(DEVICE)

#     Xt  = _t(X_tr);  yt  = _t(y_tr);  wt_t = _t(wt)
#     Xv  = _t(X_val); Xte = _t(X_test)
#     sv_tr_bin  = [ensure_binary(sv) for sv in sv_tr_list]
#     sv_te_bin  = [ensure_binary(sv) for sv in sv_te_list]
#     sv_val_bin = [ensure_binary(sv) for sv in sv_val_list]
#     st_list    = [_t(sv) for sv in sv_tr_bin]

#     encoder   = Encoder(X_tr.shape[1], p["hidden_dim"], p["dropout"]).to(DEVICE)
#     task_head = TaskHead(rep_dim).to(DEVICE)
#     encoder.load_state_dict(enc_sd); task_head.load_state_dict(th_sd)
#     bce = nn.BCEWithLogitsLoss(reduction="none")

#     base_adv_heads = nn.ModuleList([
#         AdversaryHead(rep_dim, p["hidden_dim"], p["dropout"]).to(DEVICE)
#         for _ in attr_names])

#     inter_adv_heads  = nn.ModuleList()
#     inter_target_mem = []   # list of (mi_tr_np, mj_tr_np) float32 arrays

#     def _total_heads():
#         return n_base + len(inter_adv_heads)

#     def _make_adv_opt():
#         all_p = list(base_adv_heads.parameters()) + list(inter_adv_heads.parameters())
#         return optim.AdamW(all_p, lr=p["lr"] * p["adv_lr_mult"], weight_decay=1e-4)

#     opt_enc = optim.AdamW(
#         list(encoder.parameters()) + list(task_head.parameters()),
#         lr=p["lr"], weight_decay=1e-4)
#     opt_adv = _make_adv_opt()
#     sched   = optim.lr_scheduler.CosineAnnealingLR(
#         opt_enc, p["epochs"], eta_min=p["lr"]*0.01)

#     idx_t  = torch.arange(len(y_tr), dtype=torch.long)
#     loader = _make_loader([Xt, yt, wt_t, idx_t] + st_list, p["batch"], seed)

#     best_score = np.inf; best_enc = copy.deepcopy(encoder.state_dict())
#     best_th_sd = copy.deepcopy(task_head.state_dict())
#     no_imp = 0; plateau_count = 0; best_sg_seen = np.inf; auditor_fires = 0

#     # per-epoch wall-time tracking for the paper's compute table
#     epoch_times = []

#     pbar = tqdm(range(p["epochs"]), desc="    │ [agad]",
#                 dynamic_ncols=True, leave=False, disable=not verbose)

#     for epoch in pbar:
#         t_ep = time.time()
#         grl_alpha = grl_max * max(0.1, float(epoch) / max(1, p["epochs"]))
#         adv_steps = (max(1, int(p["adv_steps"] * epoch / p["warmup_adv_epochs"]))
#                      if epoch < p["warmup_adv_epochs"] else p["adv_steps"])
#         for h in base_adv_heads: h.set_alpha(grl_alpha)
#         for h in inter_adv_heads: h.set_alpha(grl_alpha)
#         encoder.train(); task_head.train()
#         base_adv_heads.train()
#         if inter_adv_heads: inter_adv_heads.train()

#         for batch in loader:
#             xb        = batch[0]; yb = batch[1]; wb = batch[2]
#             bidx      = batch[3]
#             sv_b_list = [batch[4+i] for i in range(n_base)]
#             bidx_cpu  = bidx.cpu().numpy()
#             n_heads   = _total_heads()

#             # ── Step 1: train adversary heads on detached z (bypass GRL) ──
#             for _ in range(adv_steps):
#                 z_d = encoder(xb).detach()
#                 # .squeeze(1): AdversaryHead.net ends with Linear(*,1) → [B,1];
#                 # calling .net directly skips forward()'s squeeze, so we do it here.
#                 a_loss = sum(
#                     nn.functional.binary_cross_entropy_with_logits(
#                         base_adv_heads[i].net(z_d).squeeze(1), sv_b_list[i].float())
#                     for i in range(n_base))
#                 for k, (mi_np, mj_np) in enumerate(inter_target_mem):
#                     if k >= len(inter_adv_heads): break
#                     in_either = torch.tensor(
#                         (mi_np[bidx_cpu] + mj_np[bidx_cpu]).clip(0, 1),
#                         dtype=torch.bool, device=DEVICE)
#                     if in_either.sum() < 4: continue
#                     tgt = torch.tensor(
#                         mi_np[bidx_cpu], dtype=torch.float32, device=DEVICE)
#                     a_loss = a_loss + nn.functional.binary_cross_entropy_with_logits(
#                         inter_adv_heads[k].net(z_d).squeeze(1)[in_either], tgt[in_either])
#                 opt_adv.zero_grad(set_to_none=True)
#                 (a_loss / n_heads).backward()
#                 nn.utils.clip_grad_norm_(
#                     list(base_adv_heads.parameters()) +
#                     list(inter_adv_heads.parameters()), 1.0)
#                 opt_adv.step()

#             # ── Step 2: train encoder + task head ──
#             opt_enc.zero_grad(set_to_none=True)
#             z = encoder(xb); logits = task_head(z); prob = torch.sigmoid(logits)

#             task_loss = (bce(logits, yb) * wb).mean()

#             eo_loss = sum(_soft_eo(prob, sv_b, yb.float(), eps)
#                           for sv_b in sv_b_list) / max(n_base, 1)

#             dp_loss = sum(_soft_dp(prob, sv_b, eps)
#                           for sv_b in sv_b_list) / max(n_base, 1)

#             grl_loss = torch.tensor(0., device=DEVICE)
#             if grl_alpha > 0:
#                 grl_loss = sum(
#                     nn.functional.binary_cross_entropy_with_logits(
#                         base_adv_heads[i](z), sv_b_list[i].float())
#                     for i in range(n_base))
#                 for k, (mi_np, mj_np) in enumerate(inter_target_mem):
#                     if k >= len(inter_adv_heads): break
#                     in_either = torch.tensor(
#                         (mi_np[bidx_cpu] + mj_np[bidx_cpu]).clip(0, 1),
#                         dtype=torch.bool, device=DEVICE)
#                     if in_either.sum() < 4: continue
#                     tgt = torch.tensor(
#                         mi_np[bidx_cpu], dtype=torch.float32, device=DEVICE)
#                     # inter_adv_heads[k](z) calls forward() which already squeezes
#                     grl_loss = grl_loss + nn.functional.binary_cross_entropy_with_logits(
#                         inter_adv_heads[k](z)[in_either], tgt[in_either])
#                 grl_loss = grl_loss / n_heads

#             loss = task_lw * task_loss + fc * eo_loss + dp_fc * dp_loss + grl_loss
#             loss.backward()
#             nn.utils.clip_grad_norm_(
#                 list(encoder.parameters()) + list(task_head.parameters()), 0.5)
#             opt_enc.step()
#         sched.step()
#         epoch_times.append(time.time() - t_ep)

#         if epoch % p["eval_every"] == 0:
#             encoder.eval(); task_head.eval()
#             with torch.no_grad():
#                 pv = torch.sigmoid(task_head(encoder(Xv))).cpu().numpy()
#             val_acc = float(accuracy_score(y_val, (pv > 0.5).astype(int)))
#             val_auc = float(safe_auc(y_val, pv))
#             val_sg, _, _ = compute_eo_subgroups(
#                 y_val, pv, sv_val_bin, attr_names, min_frac, max_frac)
#             _, val_dp_wc = compute_dp_multi(pv, sv_val_bin, attr_names)

#             if val_sg < best_sg_seen - 1e-4:
#                 best_sg_seen = val_sg; plateau_count = 0
#             else:
#                 plateau_count += 1

#             if plateau_count >= PLATEAU_PATIENCE:
#                 auditor_fires += 1
#                 tqdm.write(f"\n    │ [auditor #{auditor_fires}] ep={epoch}"
#                            f"  sg_eo={val_sg:.4f}  plateau={plateau_count}")
#                 plateau_count = 0
#                 flagged = eo_gap_auditor(
#                     y_val, pv, sv_val_bin, attr_names,
#                     sv_tr_bin, min_frac, max_frac)
#                 if flagged:
#                     n_new = min(len(flagged), TOP_K_INTERSECTIONS)
#                     inter_adv_heads  = nn.ModuleList()
#                     inter_target_mem.clear()
#                     for fi in range(n_new):
#                         memi_tr, memj_tr, eo_gap, namei, namej = flagged[fi]
#                         inter_adv_heads.append(
#                             AdversaryHead(rep_dim, p["hidden_dim"], p["dropout"]).to(DEVICE))
#                         inter_target_mem.append((memi_tr, memj_tr))
#                     opt_adv = _make_adv_opt()
#                     tqdm.write(f"    │ [auditor] spawned {len(inter_adv_heads)} intersection heads\n")
#                 else:
#                     tqdm.write(f"    │ [auditor] no violations above threshold\n")

#             score = 2.0 * val_sg + 0.5 * val_dp_wc
#             if not np.isnan(val_auc): score += PARETO_LAMBDA * (1 - val_auc)

#             if val_acc >= min_acc and score < best_score:
#                 best_score = score; best_enc = copy.deepcopy(encoder.state_dict())
#                 best_th_sd = copy.deepcopy(task_head.state_dict()); no_imp = 0
#             else:
#                 no_imp += p["eval_every"]

#             if verbose:
#                 pbar.set_postfix({
#                     "sg-EO": f"{val_sg:.4f}", "DP": f"{val_dp_wc:.4f}",
#                     "acc": f"{val_acc:.3f}", "score": f"{score:.4f}",
#                     "iheads": len(inter_adv_heads), "fires": auditor_fires})

#             if no_imp >= p["patience"] * p["eval_every"]:
#                 tqdm.write(f"    │ [agad] early stop ep={epoch}")
#                 break

#     encoder.load_state_dict(best_enc); task_head.load_state_dict(best_th_sd)
#     encoder.eval(); task_head.eval()
#     with torch.no_grad():
#         proba_te = torch.sigmoid(task_head(encoder(Xte))).cpu().numpy()
#     res = _eval_and_log(proba_te, y_test, sv_te_bin, attr_names,
#                         min_frac, max_frac, tag="agad-full")
#     res["n_inter_heads"]  = int(len(inter_adv_heads))
#     res["auditor_fires"]  = int(auditor_fires)
#     res["mean_epoch_sec"] = float(np.mean(epoch_times)) if epoch_times else 0.
#     res["total_train_sec"]= float(np.sum(epoch_times))
#     return res


# # ──────────────────────────────────────────────────────────────────────────────
# # Dataset runner
# # ──────────────────────────────────────────────────────────────────────────────

# def _run_dataset(data, ds_key):
#     cfg      = DATASET_CONFIG[ds_key]
#     p        = BASE_PARAMS[ds_key]
#     X_train    = to_dense(data["X_train_nn"])
#     X_test     = to_dense(data["X_test_nn"])
#     y_train    = np.array(data["y_train"])
#     y_test     = np.array(data["y_test"])
#     attr_names = cfg["sens_attrs"]
#     sv_tr_full = [np.array(data[f"{a}_train"]) for a in attr_names]
#     sv_te      = [np.array(data[f"{a}_test"])  for a in attr_names]

#     metric_keys = ["eo", "sg_eo", "acc", "auc", "dp", "ece",
#                    "per_attr_eo", "per_attr_dp", "subgroup_worst",
#                    "n_inter_heads", "auditor_fires",
#                    "mean_epoch_sec", "total_train_sec"]
#     results = {m: {k: [] for k in metric_keys} for m in METHODS}

#     for si, seed in enumerate(SEEDS):
#         tqdm.write(_banner(f"Seed {si+1}/{N_SEEDS}  (seed={seed})", width=72, char="─"))
#         tr_idx, val_idx = build_val_split(y_train, sv_tr_full, frac=0.15, seed=seed)
#         X_tr   = X_train[tr_idx];  X_val  = X_train[val_idx]
#         y_tr   = y_train[tr_idx];  y_val  = y_train[val_idx]
#         sv_tr  = [sv[tr_idx]  for sv in sv_tr_full]
#         sv_val = [sv[val_idx] for sv in sv_tr_full]
#         tqdm.write(f"    │ split: train={len(y_tr)}  val={len(y_val)}  test={len(y_test)}"
#                    f"  label_rate={y_tr.mean():.3f}")
#         wt = compute_label_reweights(y_tr, sv_tr)
#         tqdm.write(f"    │ reweights: min={wt.min():.3f}  max={wt.max():.3f}"
#                    f"  mean={wt.mean():.3f}")
#         enc_sd, th_sd, rep_dim = pretrain_encoder(X_tr, y_tr, wt, p, seed)

#         tqdm.write(f"    ├─ [1/{len(METHODS)}] Vanilla NN")
#         r = run_vanilla_nn(X_tr, y_tr, wt, X_val, y_val, X_test, y_test,
#                            sv_te, sv_val, attr_names, enc_sd, th_sd, rep_dim, p, cfg, seed)
#         for k in metric_keys:
#             if k in r: results["vanilla_nn"][k].append(r[k])

#         tqdm.write(f"    ├─ [2/{len(METHODS)}] Zhang / No-Auditor (GRL only)")
#         r = run_zhang_adversarial(X_tr, y_tr, wt, sv_tr, sv_val, attr_names,
#                                   X_val, y_val, X_test, y_test, sv_te,
#                                   enc_sd, th_sd, rep_dim, p, cfg, seed)
#         for k in metric_keys:
#             if k in r: results["zhang_adversarial"][k].append(r[k])

#         mi = 3
#         if HAS_FAIRLEARN:
#             tqdm.write(f"    ├─ [{mi}/{len(METHODS)}] FairLearn Adversarial")
#             r = run_fairlearn_adv(X_tr, y_tr, X_test, y_test,
#                                   sv_tr, sv_te, attr_names, p, cfg, seed)
#             if r is not None:
#                 for k in metric_keys:
#                     if k in r: results["fairlearn_adv"][k].append(r[k])
#             mi += 1

#         tqdm.write(f"    └─ [{mi}/{len(METHODS)}] AGAD Full")
#         r = _train_agad(X_tr, y_tr, wt, sv_tr, attr_names,
#                         X_val, y_val, sv_val, X_test, y_test, sv_te,
#                         enc_sd, th_sd, rep_dim, p, cfg, verbose=True, seed=seed)
#         for k in metric_keys:
#             if k in r: results["agad_full"][k].append(r[k])

#         gc.collect()
#         if torch.cuda.is_available(): torch.cuda.empty_cache()

#     return results


# # ──────────────────────────────────────────────────────────────────────────────
# # Preprocessing
# # ──────────────────────────────────────────────────────────────────────────────

# def _cache_path(name):
#     return f"/tmp/agad{CACHE_VERSION}_{name}.pkl"

# def preprocess_acs_employment(use_cache=True):
#     cache = _cache_path("acs_employment")
#     if use_cache and os.path.exists(cache): return joblib.load(cache)
#     if not HAS_FOLKTABLES: raise RuntimeError("pip install folktables")
#     data_source = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
#     acs_data    = data_source.get_data(states=ACS_STATES, download=True)
#     features, label, group = ACSEmployment.df_to_numpy(acs_data)
#     valid    = ~np.isnan(label.astype(float))
#     features = features[valid].astype(np.float32); label = label[valid].astype(int)
#     acs_elig = acs_data[valid].reset_index(drop=True)
#     RACE_MAP_4  = {1: 0, 2: 1, 3: 3, 4: 2, 5: 2, 6: 3, 7: 3, 8: 3, 9: 3}
#     RACE_NAMES4 = ["White", "Black", "Asian_PI", "Other"]
#     rac1p = acs_elig["RAC1P"].values; hisp = acs_elig["HISP"].values
#     race  = np.array([RACE_MAP_4.get(int(r), 3) for r in rac1p])
#     race[hisp != 1] = 3
#     sex = (acs_elig["SEX"].values == 2).astype(int)
#     tr, te = stratified_split(features, label, [race, sex])
#     scaler = StandardScaler()
#     X_tr_s = scaler.fit_transform(features[tr]); X_te_s = scaler.transform(features[te])
#     res = {"X_train_nn": X_tr_s, "X_test_nn": X_te_s,
#            "y_train": label[tr], "y_test": label[te]}
#     for g, nm in enumerate(RACE_NAMES4):
#         arr_tr = (race[tr] == g).astype(int); arr_te = (race[te] == g).astype(int)
#         if arr_te.mean() >= 0.02:
#             res[f"race_{nm}_train"] = arr_tr; res[f"race_{nm}_test"] = arr_te
#         else:
#             tqdm.write(f"  [exclude] race_{nm} = {arr_te.mean():.3f} — too small")
#     res["sex_Female_train"] = sex[tr]; res["sex_Female_test"] = sex[te]
#     DATASET_CONFIG["acs_employment"]["sens_attrs"] = [
#         k.replace("_train", "") for k in res
#         if k.endswith("_train") and k not in ("y_train", "X_train_nn")]
#     joblib.dump(res, cache); return res

# def preprocess_adult(use_cache=True):
#     cache = _cache_path("adult")
#     if use_cache and os.path.exists(cache): return joblib.load(cache)
#     from ucimlrepo import fetch_ucirepo
#     repo  = fetch_ucirepo(id=2)
#     X_df  = repo.data.features.copy(); y_ser = repo.data.targets.copy()
#     y_raw      = y_ser.iloc[:, 0].astype(str).str.strip().str.rstrip(".")
#     y          = (y_raw == ">50K").astype(int).values
#     race_raw   = X_df["race"].astype(str).str.strip()
#     race_Black = (race_raw == "Black").astype(int).values
#     race_White = (race_raw == "White").astype(int).values
#     sex        = (X_df["sex"].astype(str).str.strip() == "Female").astype(int).values
#     drop_cols  = ["race", "sex", "fnlwgt", "education-num"]
#     X_df = X_df.drop(columns=[c for c in drop_cols if c in X_df.columns])
#     X_df = pd.get_dummies(X_df); X = X_df.values.astype(np.float32)
#     valid = (~np.isnan(X).any(axis=1)) & (~pd.isna(y_raw).values)
#     X, y = X[valid], y[valid]
#     race_Black = race_Black[valid]; race_White = race_White[valid]; sex = sex[valid]
#     tr, te = stratified_split(X, y, [race_Black, sex])
#     scaler = StandardScaler()
#     X_tr_s = scaler.fit_transform(X[tr]); X_te_s = scaler.transform(X[te])
#     res = {"X_train_nn": X_tr_s, "X_test_nn": X_te_s,
#            "y_train": y[tr], "y_test": y[te],
#            "race_Black_train": race_Black[tr], "race_Black_test": race_Black[te],
#            "race_White_train": race_White[tr], "race_White_test": race_White[te],
#            "sex_Female_train": sex[tr],        "sex_Female_test": sex[te]}
#     joblib.dump(res, cache); return res

# def preprocess_communities_crime(use_cache=True):
#     cache = _cache_path("communities_crime")
#     if use_cache and os.path.exists(cache): return joblib.load(cache)
#     from ucimlrepo import fetch_ucirepo
#     repo  = fetch_ucirepo(id=183)
#     X_df  = repo.data.features.copy(); y_df = repo.data.targets.copy()
#     y_cont = pd.to_numeric(y_df.iloc[:, 0], errors="coerce").values
#     valid  = ~np.isnan(y_cont); y_cont = y_cont[valid]
#     X_df   = X_df[valid].reset_index(drop=True)
#     y      = (y_cont > np.median(y_cont)).astype(int)
#     def find_col(df, patterns):
#         for pat in patterns:
#             matches = [c for c in df.columns if pat.lower() in c.lower()]
#             if matches: return pd.to_numeric(df[matches[0]], errors="coerce")
#         return None
#     col_black  = find_col(X_df, ["racepctblack", "pctblack"])
#     col_income = find_col(X_df, ["medIncome", "medincome"])
#     col_unemp  = find_col(X_df, ["PctUnemployed", "pctunemployed"])
#     def binarise(col, high_is_1=True):
#         if col is None: return None
#         col = col.fillna(col.median()).values; med = np.median(col)
#         return (col > med).astype(int) if high_is_1 else (col < med).astype(int)
#     s_black  = binarise(col_black,  high_is_1=True)
#     s_income = binarise(col_income, high_is_1=False)
#     s_unemp  = binarise(col_unemp,  high_is_1=True)
#     # Guard against missing columns — fall back to all-zeros (no signal)
#     n = len(y)
#     if s_black  is None: tqdm.write("  [warn] racepctblack not found"); s_black  = np.zeros(n, dtype=int)
#     if s_income is None: tqdm.write("  [warn] medIncome not found");    s_income = np.zeros(n, dtype=int)
#     if s_unemp  is None: tqdm.write("  [warn] PctUnemployed not found");s_unemp  = np.zeros(n, dtype=int)
#     X_df_num  = X_df.apply(pd.to_numeric, errors="coerce")
#     X_df_num  = X_df_num.dropna(axis=1, thresh=int(0.7*len(X_df_num)))
#     X_df_num  = X_df_num.fillna(X_df_num.median())
#     sens_drop = ["racepct", "racePct", "medIncome", "PctUnemployed",
#                  "pctFemale", "ViolentCrimesPerPop"]
#     drop_cols = [c for c in X_df_num.columns
#                  if any(pt.lower() in c.lower() for pt in sens_drop)]
#     X = X_df_num.drop(columns=drop_cols, errors="ignore").values.astype(np.float32)
#     tr, te = stratified_split(X, y, [s_black, s_income, s_unemp])
#     scaler = StandardScaler()
#     X_tr_s = scaler.fit_transform(X[tr]); X_te_s = scaler.transform(X[te])
#     res = {"X_train_nn": X_tr_s, "X_test_nn": X_te_s,
#            "y_train": y[tr], "y_test": y[te],
#            "high_black_pct_train":    s_black[tr],  "high_black_pct_test":    s_black[te],
#            "low_income_train":        s_income[tr], "low_income_test":        s_income[te],
#            "high_unemployment_train": s_unemp[tr],  "high_unemployment_test": s_unemp[te]}
#     joblib.dump(res, cache); return res

# PREPROCESS_FNS = {
#     "acs_employment":    preprocess_acs_employment,
#     "adult":             preprocess_adult,
#     "communities_crime": preprocess_communities_crime,
# }


# # ──────────────────────────────────────────────────────────────────────────────
# # Results printing
# # ──────────────────────────────────────────────────────────────────────────────

# DS_LABELS = {
#     "acs_employment":    "ACS Employment",
#     "adult":             "Adult Income",
#     "communities_crime": "Communities & Crime",
# }

# def _stats(mr, m, key):
#     vals = [_to_float(v) for v in mr.get(m, {}).get(key, [])]
#     vals = [v for v in vals if v is not None]
#     if not vals: return float("nan"), float("nan")
#     return float(np.mean(vals)), float(np.std(vals))

# def print_results(all_results):
#     W = 128
#     print(f"\n{'═'*W}")
#     print(f"  AGAD v17  |  SG-EO <{TARGET_EO}  DP <{TARGET_DP}"
#           f"  |  seeds={SEEDS}  N={N_SEEDS}"
#           f"  |  device={DEVICE}")
#     print(f"  Methods: vanilla_nn → zhang(no-auditor) → fairlearn_adv → agad_full")
#     print(f"  AGAD extras vs zhang: soft-EO (coef={BASE_PARAMS['adult']['fairness_coef']}†)"
#           f"  soft-DP ({DP_COEF_FRAC}×coef)"
#           f"  auditor(thresh={EO_GAP_FIRE_THRESH}, top_k={TOP_K_INTERSECTIONS},"
#           f" plateau={PLATEAU_PATIENCE})")
#     print(f"  † coef varies per dataset; see BASE_PARAMS")
#     print(f"{'─'*W}")

#     for ds_key, mr in all_results.items():
#         print(f"\n  {'▶ ' + DS_LABELS.get(ds_key, ds_key).upper()}")
#         hdr = (f"  {'Method':<24} {'WC-EO':>8} {'SG-EO':>10} {'SG-EO±':>8} {'WC-DP':>8}"
#                f" {'Acc':>8} {'Acc±':>7} {'AUC':>8} {'ECE':>7}  EO   DP")
#         print(hdr); print(f"  {'─'*W}")
#         for m in METHODS:
#             if m not in mr or not mr[m].get("sg_eo"): continue
#             eo_m, _    = _stats(mr, m, "eo")
#             sg_m, sg_s = _stats(mr, m, "sg_eo")
#             dp_m, _    = _stats(mr, m, "dp")
#             ac_m, ac_s = _stats(mr, m, "acc")
#             au_m, _    = _stats(mr, m, "auc")
#             ec_m, _    = _stats(mr, m, "ece")
#             n_h = int(np.nanmean([v for v in mr[m].get("n_inter_heads", [])
#                                   if v is not None] or [0]))
#             af  = int(np.nanmean([v for v in mr[m].get("auditor_fires", [])
#                                   if v is not None] or [0]))
#             def _f(v): return f"{v:8.4f}" if not np.isnan(v) else "     nan"
#             def _fs(v): return f"{v:7.4f}" if not np.isnan(v) else "    nan"
#             s_eo = "PASS" if (not np.isnan(sg_m) and sg_m < TARGET_EO) else "FAIL"
#             s_dp = "PASS" if (not np.isnan(dp_m) and dp_m < TARGET_DP)  else "FAIL"
#             label = METHOD_LABELS.get(m, m)
#             extra = f"  iheads={n_h}  fires={af}" if m == "agad_full" else ""
#             print(f"  {label:<24} {_f(eo_m)} {_f(sg_m)} {_fs(sg_s)} {_f(dp_m)}"
#                   f" {_f(ac_m)} {_fs(ac_s)} {_f(au_m)} {_f(ec_m)}  {s_eo}  {s_dp}{extra}")

#         if "agad_full" in mr and "zhang_adversarial" in mr:
#             sg_z, _ = _stats(mr, "zhang_adversarial", "sg_eo")
#             sg_a, _ = _stats(mr, "agad_full",         "sg_eo")
#             dp_z, _ = _stats(mr, "zhang_adversarial", "dp")
#             dp_a, _ = _stats(mr, "agad_full",         "dp")
#             if not np.isnan(sg_z) and not np.isnan(sg_a):
#                 tag = "↓ improved" if sg_z - sg_a > 0.002 else ("~ marginal" if sg_z > sg_a else "= flat")
#                 print(f"\n    Auditor gain:  SG-EO {sg_z:.4f}→{sg_a:.4f} (Δ={sg_a-sg_z:+.4f} {tag})"
#                       f"   DP {dp_z:.4f}→{dp_a:.4f} (Δ={dp_a-dp_z:+.4f})")

#     print(f"\n{'═'*W}")
#     print(f"  PASS/FAIL MATRIX  (mean±std over {N_SEEDS} seeds"
#           f"  EO<{TARGET_EO}  DP<{TARGET_DP})\n")
#     hdr2 = f"  {'Dataset':<26}" + "".join(f" {METHOD_LABELS.get(m,'?'):>26}" for m in METHODS)
#     print(hdr2); print(f"  {'─'*W}")
#     for ds_key, mr in all_results.items():
#         row = f"  {DS_LABELS.get(ds_key, ds_key):<26}"
#         for m in METHODS:
#             sg, sg_s = _stats(mr, m, "sg_eo"); dp, _ = _stats(mr, m, "dp")
#             if np.isnan(sg): cell = "         N/A         "
#             else:
#                 eo_p = "EO✓" if sg < TARGET_EO else "EO✗"
#                 dp_p = "DP✓" if dp < TARGET_DP else "DP✗"
#                 cell = f"{eo_p} {dp_p} {sg:.3f}±{sg_s:.3f}/{dp:.3f}"
#             row += f" {cell:>26}"
#         print(row)

#     # Compute table for paper
#     print(f"\n{'─'*W}")
#     print(f"  COMPUTE (AGAD only, mean over seeds)\n")
#     print(f"  {'Dataset':<26} {'mean_epoch_sec':>16} {'total_train_sec':>17}")
#     for ds_key, mr in all_results.items():
#         me, _ = _stats(mr, "agad_full", "mean_epoch_sec")
#         tt, _ = _stats(mr, "agad_full", "total_train_sec")
#         def _fc(v): return f"{v:>16.1f}" if not np.isnan(v) else "             nan"
#         print(f"  {DS_LABELS.get(ds_key, ds_key):<26} {_fc(me)} {_fc(tt)}")
#     print(f"{'═'*W}\n")


# # ──────────────────────────────────────────────────────────────────────────────
# # Plotting
# # ──────────────────────────────────────────────────────────────────────────────

# def plot_results(all_results, save_dir="/kaggle/working"):
#     try:
#         import matplotlib
#         matplotlib.use("Agg")
#         import matplotlib.pyplot as plt
#         import matplotlib.patches as mpatches
#     except ImportError:
#         tqdm.write("  [plot] matplotlib not available — skipping plots")
#         return []

#     os.makedirs(save_dir, exist_ok=True)
#     saved = []
#     active_methods = [m for m in METHODS if any(
#         mr.get(m, {}).get("sg_eo") for mr in all_results.values())]
#     n_m  = len(active_methods)
#     n_ds = len(all_results)

#     def _mean_std(mr, m, key):
#         vals = [_to_float(v) for v in mr.get(m, {}).get(key, [])]
#         vals = [v for v in vals if v is not None]
#         if not vals: return float("nan"), 0.
#         return float(np.mean(vals)), float(np.std(vals))

#     metrics_cfg = [
#         ("sg_eo", "SG-EO (↓ fairer)",   TARGET_EO, True),
#         ("dp",    "WC-DP (↓ fairer)",   TARGET_DP, True),
#         ("acc",   "Accuracy (↑ better)", None,      False),
#     ]
#     fig, axes = plt.subplots(n_ds, 3, figsize=(13, 3.8 * n_ds),
#                               gridspec_kw={"hspace": 0.55, "wspace": 0.38})
#     if n_ds == 1: axes = axes[np.newaxis, :]
#     bar_w = 0.65 / n_m

#     for di, (ds_key, mr) in enumerate(all_results.items()):
#         for ci, (mkey, mlabel, target, lower_better) in enumerate(metrics_cfg):
#             ax = axes[di, ci]
#             for mi, m in enumerate(active_methods):
#                 mean, std = _mean_std(mr, m, mkey)
#                 if np.isnan(mean): continue
#                 color = METHOD_COLORS.get(m, "#aaa")
#                 ax.bar(mi, mean, bar_w * 0.88, yerr=std, capsize=3,
#                        color=color, alpha=0.85, edgecolor="white", linewidth=0.8)
#                 seed_vals = [_to_float(v) for v in mr.get(m, {}).get(mkey, [])]
#                 seed_vals = [v for v in seed_vals if v is not None]
#                 ax.scatter([mi]*len(seed_vals), seed_vals,
#                            color="white", s=18, zorder=5, linewidths=0.8,
#                            edgecolors=color)
#             if target is not None:
#                 ax.axhline(target, color="crimson", linestyle="--",
#                            linewidth=1.3, alpha=0.8, label=f"target {target}")
#                 ax.legend(fontsize=6.5, loc="upper right")
#             ax.set_xticks(range(n_m))
#             ax.set_xticklabels(
#                 [METHOD_LABELS.get(m, m).replace(" ", "\n") for m in active_methods],
#                 fontsize=7)
#             ax.set_ylabel(mlabel, fontsize=8)
#             ax.set_title(f"{DS_LABELS.get(ds_key, ds_key)}", fontsize=8.5, pad=4)
#             ax.grid(axis="y", alpha=0.25, linewidth=0.7)
#             ax.spines["top"].set_visible(False)
#             ax.spines["right"].set_visible(False)

#     legend_patches = [mpatches.Patch(color=METHOD_COLORS.get(m, "#aaa"),
#                                      label=METHOD_LABELS.get(m, m))
#                       for m in active_methods]
#     fig.legend(handles=legend_patches, loc="upper center", ncol=n_m,
#                fontsize=8, bbox_to_anchor=(0.5, 1.01), frameon=False)
#     p1 = os.path.join(save_dir, "results_bar.png")
#     fig.savefig(p1, dpi=160, bbox_inches="tight"); plt.close(fig)
#     tqdm.write(f"  [plot] → {p1}"); saved.append(p1)

#     # Pareto scatter
#     fig2, axes2 = plt.subplots(1, n_ds, figsize=(5.2 * n_ds, 4.5),
#                                 gridspec_kw={"wspace": 0.35})
#     if n_ds == 1: axes2 = [axes2]
#     markers = {"vanilla_nn": "o", "zhang_adversarial": "s",
#                 "fairlearn_adv": "^", "agad_full": "D"}
#     for di, (ds_key, mr) in enumerate(all_results.items()):
#         ax = axes2[di]
#         for m in active_methods:
#             sg_vals = [_to_float(v) for v in mr.get(m, {}).get("sg_eo", [])]
#             ac_vals = [_to_float(v) for v in mr.get(m, {}).get("acc",   [])]
#             sg_vals = [v for v in sg_vals if v is not None]
#             ac_vals = [v for v in ac_vals if v is not None]
#             n = min(len(sg_vals), len(ac_vals))
#             if n == 0: continue
#             ax.scatter(sg_vals[:n], ac_vals[:n],
#                        color=METHOD_COLORS.get(m, "#aaa"),
#                        marker=markers.get(m, "o"), s=28, alpha=0.45, zorder=3)
#             ax.scatter(np.mean(sg_vals[:n]), np.mean(ac_vals[:n]),
#                        color=METHOD_COLORS.get(m, "#aaa"),
#                        marker=markers.get(m, "o"), s=110, zorder=5,
#                        edgecolors="black", linewidths=0.8,
#                        label=METHOD_LABELS.get(m, m))
#         ax.axvline(TARGET_EO, color="crimson", linestyle="--",
#                    linewidth=1.2, alpha=0.75, label=f"EO target={TARGET_EO}")
#         ax.set_xlabel("SG-EO (lower = fairer)", fontsize=9)
#         ax.set_ylabel("Accuracy", fontsize=9)
#         ax.set_title(DS_LABELS.get(ds_key, ds_key), fontsize=10, pad=5)
#         ax.legend(fontsize=7, framealpha=0.8)
#         ax.grid(alpha=0.25); ax.spines["top"].set_visible(False)
#         ax.spines["right"].set_visible(False)
#     fig2.suptitle("Accuracy–Fairness Pareto Frontier  (large = mean, small = per-seed)",
#                   fontsize=10, y=1.01)
#     p2 = os.path.join(save_dir, "pareto_scatter.png")
#     fig2.savefig(p2, dpi=160, bbox_inches="tight"); plt.close(fig2)
#     tqdm.write(f"  [plot] → {p2}"); saved.append(p2)

#     # Per-attribute EO heatmap
#     # per_attr_eo is a list of floats (one per seed), shape [n_seeds, n_attrs]
#     try:
#         fig3, axes3 = plt.subplots(1, n_ds, figsize=(5.5 * n_ds, 3.8),
#                                     gridspec_kw={"wspace": 0.4})
#         if n_ds == 1: axes3 = [axes3]
#         for di, (ds_key, mr) in enumerate(all_results.items()):
#             ax = axes3[di]
#             attr_names = DATASET_CONFIG[ds_key]["sens_attrs"]
#             rows = []
#             for m in active_methods:
#                 if m not in mr or not mr[m].get("per_attr_eo"): continue
#                 pa = mr[m]["per_attr_eo"]
#                 # pa is a list of per-seed values, each of which is a list of floats
#                 pa_arr = np.array([x for x in pa if isinstance(x, (list, np.ndarray))])
#                 if pa_arr.size == 0: continue
#                 rows.append((METHOD_LABELS.get(m, m), pa_arr.mean(axis=0)))
#             if not rows: continue
#             labels_r, data_r = zip(*rows)
#             mat = np.array(data_r)
#             im = ax.imshow(mat, aspect="auto", cmap="YlOrRd",
#                            vmin=0, vmax=min(0.3, mat.max() * 1.1))
#             ax.set_xticks(range(len(attr_names)))
#             ax.set_xticklabels([a.replace("_", "\n") for a in attr_names], fontsize=7)
#             ax.set_yticks(range(len(labels_r)))
#             ax.set_yticklabels(labels_r, fontsize=7)
#             for ri in range(mat.shape[0]):
#                 for ci2 in range(mat.shape[1]):
#                     ax.text(ci2, ri, f"{mat[ri,ci2]:.3f}",
#                             ha="center", va="center", fontsize=6.5,
#                             color="black" if mat[ri,ci2] < 0.15 else "white")
#             ax.set_title(f"{DS_LABELS.get(ds_key, ds_key)}\nPer-attr WC-EO", fontsize=9)
#             fig3.colorbar(im, ax=ax, shrink=0.75, label="EO gap")
#         p3 = os.path.join(save_dir, "attr_eo_heatmap.png")
#         fig3.savefig(p3, dpi=160, bbox_inches="tight"); plt.close(fig3)
#         tqdm.write(f"  [plot] → {p3}"); saved.append(p3)
#     except Exception as e:
#         tqdm.write(f"  [plot] heatmap skipped: {e}")

#     return saved


# # ──────────────────────────────────────────────────────────────────────────────
# # Save results
# # ──────────────────────────────────────────────────────────────────────────────

# def save_results(all_results, save_dir="/kaggle/working"):
#     def _clean(obj):
#         if isinstance(obj, dict):                    return {k: _clean(v) for k, v in obj.items()}
#         if isinstance(obj, list):                    return [_clean(v) for v in obj]
#         if isinstance(obj, float) and np.isnan(obj): return None
#         if isinstance(obj, np.floating):             return float(obj)
#         if isinstance(obj, np.integer):              return int(obj)
#         if isinstance(obj, np.ndarray):              return obj.tolist()
#         if isinstance(obj, torch.Tensor):            return obj.tolist()
#         return obj
#     path = os.path.join(save_dir, "results_agad_v17.json")
#     os.makedirs(save_dir, exist_ok=True)
#     with open(path, "w") as f:
#         json.dump(_clean({"results": all_results, "seeds": SEEDS,
#                           "version": CACHE_VERSION, "n_seeds": N_SEEDS,
#                           "methods": METHODS,
#                           "hyperparams": BASE_PARAMS,
#                           "targets": {"eo": TARGET_EO, "dp": TARGET_DP},
#                           "config": {
#                               "MAX_SG_ATTRS": MAX_SG_ATTRS,
#                               "TOP_K_INTERSECTIONS": TOP_K_INTERSECTIONS,
#                               "PLATEAU_PATIENCE": PLATEAU_PATIENCE,
#                               "EO_GAP_FIRE_THRESH": EO_GAP_FIRE_THRESH,
#                               "DP_COEF_FRAC": DP_COEF_FRAC,
#                               "PARETO_LAMBDA": PARETO_LAMBDA,
#                           }}), f, indent=2)
#     tqdm.write(f"  Results → {path}")
#     return path


# # ──────────────────────────────────────────────────────────────────────────────
# # Main
# # ──────────────────────────────────────────────────────────────────────────────

# def main():
#     t0 = time.time()
#     tqdm.write(_banner(f"AGAD v17  |  device={DEVICE}  seeds={SEEDS}"
#                        f"  fairlearn={'yes' if HAS_FAIRLEARN else 'NO (pip install fairlearn)'}",
#                        width=80))
#     tqdm.write(f"  Methods: {METHODS}\n")
#     all_results = {}
#     for ds_key in RUN_DATASETS:
#         tqdm.write(_banner(f"DATASET: {ds_key.upper()}", width=80))
#         data = PREPROCESS_FNS[ds_key]()
#         cfg  = DATASET_CONFIG[ds_key]
#         y_te = np.array(data["y_test"])
#         sv_p = ensure_binary(np.array(data[f"{cfg['sens_attrs'][0]}_test"]))
#         tqdm.write(f"  n_train={len(data['y_train'])}  n_test={len(y_te)}"
#                    f"  label_pos={y_te.mean():.3f}  sens_pos={sv_p.mean():.3f}")
#         tqdm.write(f"  attrs: {cfg['sens_attrs']}\n")
#         ds_t0 = time.time()
#         all_results[ds_key] = _run_dataset(data, ds_key)
#         tqdm.write(f"  [{ds_key}] wall time: {(time.time()-ds_t0)/60:.1f} min")
#         del data; gc.collect()
#         if torch.cuda.is_available(): torch.cuda.empty_cache()

#     print_results(all_results)
#     plots = plot_results(all_results)
#     json_path = save_results(all_results)
#     elapsed = (time.time() - t0) / 60
#     tqdm.write(f"\n  Total wall time: {elapsed:.1f} min")
#     tqdm.write(f"  Outputs: {json_path}  +  {len(plots)} plots")
#     return all_results


# if __name__ == "__main__":
#     main()

In [3]:
import os, gc, warnings, json, time, copy, random
os.environ.setdefault("PYTHONHASHSEED", "42")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

warnings.filterwarnings("ignore")
torch.use_deterministic_algorithms(True, warn_only=False)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except ImportError:
    HAS_OPTUNA = False

try:
    from folktables import ACSDataSource, ACSEmployment
    HAS_FOLKTABLES = True
except ImportError:
    HAS_FOLKTABLES = False

try:
    from fairlearn.adversarial import AdversarialFairnessClassifier
    HAS_FAIRLEARN = True
except ImportError:
    HAS_FAIRLEARN = False

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

SEED          = 42
N_SEEDS       = 3
SEEDS         = list(range(SEED, SEED + N_SEEDS))
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TARGET_EO     = 0.05
TARGET_DP     = 0.10
CACHE_VERSION = "v19"

ACS_STATES          = ["CA"]
TOP_K_INTERSECTIONS = 5
MAX_SG_ATTRS        = 2
PLATEAU_PATIENCE    = 2
PARETO_LAMBDA       = 0.10
EO_GAP_FIRE_THRESH  = 0.015

OPTUNA_N_TRIALS_PER_DS = 25
OPTUNA_SEED_SINGLE     = SEED
OPTUNA_EPOCHS_FRAC     = 0.45
OPTUNA_MAX_TRAIN_ROWS  = 40000

RUN_DIAGNOSTIC_SWEEP = True
DIAG_SWEEP_POINTS    = 5

RUN_DATASETS = ["acs_employment", "adult", "communities_crime"]

DATASET_CONFIG = {
    "acs_employment": {
        "sens_attrs":      ["race_White", "race_Black", "race_Other", "sex_Female"],
        "reweight_labels": True,
        "min_frac": 0.02, "max_frac": 0.40,
    },
    "adult": {
        "sens_attrs":      ["race_Black", "race_White", "sex_Female"],
        "reweight_labels": True,
        "min_frac": 0.02, "max_frac": 0.40,
    },
    "communities_crime": {
        "sens_attrs":      ["high_black_pct", "low_income", "high_unemployment"],
        "reweight_labels": True,
        "min_frac": 0.05, "max_frac": 0.45,
    },
}

BASE_PARAMS = {
    "acs_employment": {
        "hidden_dim": 96, "dropout": 0.15, "lr": 5e-4,
        "batch": 4096, "epochs": 55,
        "eo_coef": 20.0, "dp_coef": 10.0,
        "eval_every": 4, "patience": 16,
        "min_acc_floor": 0.79,
        "adv_lr_mult": 4.0, "adv_steps": 3,
        "pre_epochs": 8,
        "task_lw": 2.5,
    },
    "adult": {
        "hidden_dim": 96, "dropout": 0.15, "lr": 5e-4,
        "batch": 2048, "epochs": 65,
        "eo_coef": 30.0, "dp_coef": 15.0,
        "eval_every": 4, "patience": 18,
        "min_acc_floor": 0.83,
        "adv_lr_mult": 5.0, "adv_steps": 4,
        "pre_epochs": 10,
        "task_lw": 2.0,
    },
    "communities_crime": {
        "hidden_dim": 64, "dropout": 0.20, "lr": 3e-4,
        "batch": 256, "epochs": 140,
        "eo_coef": 60.0, "dp_coef": 30.0,
        "eval_every": 4, "patience": 32,
        "min_acc_floor": 0.72,
        "adv_lr_mult": 9.0, "adv_steps": 8,
        "pre_epochs": 12,
        "task_lw": 1.0,
    },
}

ZHANG_PARAMS = {
    "acs_employment": {
        "hidden_dim": 96, "dropout": 0.15, "lr": 5e-4,
        "batch": 4096, "epochs": 55,
        "eval_every": 4, "patience": 16,
        "min_acc_floor": 0.68,
        "adv_lr_mult": 4.0, "adv_steps": 3,
        "pre_epochs": 8, "task_lw": 2.5, "grl_max": 0.65,
    },
    "adult": {
        "hidden_dim": 96, "dropout": 0.15, "lr": 5e-4,
        "batch": 2048, "epochs": 65,
        "eval_every": 4, "patience": 18,
        "min_acc_floor": 0.72,
        "adv_lr_mult": 5.0, "adv_steps": 4,
        "pre_epochs": 10, "task_lw": 2.0, "grl_max": 0.80,
    },
    "communities_crime": {
        "hidden_dim": 64, "dropout": 0.20, "lr": 3e-4,
        "batch": 256, "epochs": 140,
        "eval_every": 4, "patience": 32,
        "min_acc_floor": 0.50,
        "adv_lr_mult": 9.0, "adv_steps": 8,
        "pre_epochs": 12, "task_lw": 1.0, "grl_max": 0.95,
    },
}

TUNED_PARAMS = {}
AUDITOR_LOG  = {}

METHOD_LABELS = {
    "vanilla_nn":        "Vanilla NN",
    "zhang_adversarial": "Zhang (GRL-only)",
    "fairlearn_adv":     "FairLearn Adv",
    "agad_full":         "AGAD (full)",
}
METHOD_COLORS = {
    "vanilla_nn":        "#e74c3c",
    "zhang_adversarial": "#f39c12",
    "fairlearn_adv":     "#2ecc71",
    "agad_full":         "#3498db",
}

DS_LABELS = {
    "acs_employment":    "ACS Employment",
    "adult":             "Adult Income",
    "communities_crime": "Communities & Crime",
}

METHODS = ["vanilla_nn", "zhang_adversarial"]
if HAS_FAIRLEARN:
    METHODS.append("fairlearn_adv")
METHODS.append("agad_full")


def set_seed(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

def to_dense(X):
    return X.toarray() if hasattr(X, "toarray") else np.array(X)

def ensure_binary(sv):
    sv = np.array(sv).flatten()
    u  = np.unique(sv[~np.isnan(sv.astype(float))])
    if len(u) <= 1: return np.zeros(len(sv), dtype=int)
    if len(u) == 2: return (sv == u[1]).astype(int)
    maj = u[np.argmax([(sv == v).sum() for v in u])]
    return (sv != maj).astype(int)

def stratified_split(X, y, sens_list, test_size=0.2):
    key = np.array(y).astype(str).copy()
    for s in sens_list:
        key = key + "_" + np.array(s).astype(str)
    u, c = np.unique(key, return_counts=True)
    key[np.isin(key, u[c < 2])] = np.array(y)[np.isin(key, u[c < 2])].astype(str)
    try:
        return train_test_split(np.arange(len(y)), test_size=test_size,
                                stratify=key, random_state=SEED)
    except Exception:
        return train_test_split(np.arange(len(y)), test_size=test_size,
                                stratify=y, random_state=SEED)

def build_val_split(y_train, sv_list, frac=0.15, seed=SEED):
    key = np.array(y_train).astype(str)
    for sv in sv_list:
        key = key + "_" + ensure_binary(np.array(sv)).astype(str)
    u, c = np.unique(key, return_counts=True)
    key[np.isin(key, u[c < 2])] = np.array(y_train)[np.isin(key, u[c < 2])].astype(str)
    try:
        return train_test_split(np.arange(len(y_train)), test_size=frac,
                                stratify=key, random_state=seed)
    except Exception:
        return train_test_split(np.arange(len(y_train)), test_size=frac,
                                stratify=y_train, random_state=seed)

def safe_auc(y_true, y_score):
    try:
        if len(np.unique(y_true)) < 2: return float("nan")
        return roc_auc_score(y_true, y_score)
    except Exception:
        return float("nan")

def _to_float(v):
    try:
        f = float(v)
        return None if (f != f) else f
    except Exception:
        return None

def _banner(title, width=80, char="═"):
    pad = max(0, width - len(title) - 4)
    return f"\n  {char*2} {title} {char * pad}"


def compute_label_reweights(y, sv_list):
    y    = np.array(y)
    n    = len(y)
    p_y1 = np.clip(y.mean(), 1e-3, 1-1e-3)
    p_y0 = 1.0 - p_y1
    w_sum = np.zeros(n, dtype=np.float64)
    count = 0
    for sv in sv_list:
        sv_b = ensure_binary(sv)
        if len(np.unique(sv_b)) < 2: continue
        w = np.ones(n, dtype=np.float64)
        for g in [0, 1]:
            mask = sv_b == g
            if mask.sum() == 0: continue
            p_y1_g = np.clip(y[mask].mean(), 1e-3, 1-1e-3)
            w[mask & (y==1)] = p_y1 / p_y1_g
            w[mask & (y==0)] = p_y0 / (1 - p_y1_g)
        w_sum += w
        count += 1
    if count == 0: return np.ones(n, dtype=np.float32)
    w_avg = w_sum / count
    return (w_avg / w_avg.mean()).astype(np.float32)


def compute_eo_binary(y_true, y_prob, sv):
    sv = ensure_binary(sv)
    if len(np.unique(sv)) != 2: return 0.0
    tprs, fprs = [], []
    for g in [0, 1]:
        pos = (sv==g) & (y_true==1)
        neg = (sv==g) & (y_true==0)
        tprs.append(float(y_prob[pos].mean()) if pos.sum() > 0 else 0.0)
        fprs.append(float(y_prob[neg].mean()) if neg.sum() > 0 else 0.0)
    return max(abs(tprs[0]-tprs[1]), abs(fprs[0]-fprs[1]))

def compute_eo_multi(y_true, y_prob, sv_list, attr_names):
    per = [compute_eo_binary(y_true, y_prob, sv) for sv in sv_list]
    return per, max(per) if per else 0.0

def compute_dp(y_prob, sv):
    sv = ensure_binary(sv)
    if len(np.unique(sv)) != 2: return 0.0
    r0 = float(y_prob[sv==0].mean()) if (sv==0).sum() > 0 else 0.5
    r1 = float(y_prob[sv==1].mean()) if (sv==1).sum() > 0 else 0.5
    return abs(r0 - r1)

def compute_dp_multi(y_prob, sv_list, attr_names):
    per = [compute_dp(y_prob, sv) for sv in sv_list]
    return per, max(per) if per else 0.0

def compute_ece(y_true, y_prob, n_bins=15):
    bins = np.linspace(0, 1, n_bins+1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() == 0: continue
        ece += mask.sum() * abs(y_true[mask].mean() - y_prob[mask].mean())
    return float(ece / len(y_true))

def _soft_eo(prob, sv_b, yf, eps):
    sf = sv_b.float()
    g0y1 = (1-sf)*yf;     g1y1 = sf*yf
    g0y0 = (1-sf)*(1-yf); g1y0 = sf*(1-yf)
    if any(gg.sum() < 1e-6 for gg in [g0y1, g1y1, g0y0, g1y0]):
        return torch.tensor(0., device=prob.device)
    tpr0 = (prob*g0y1).sum() / (g0y1.sum()+eps)
    tpr1 = (prob*g1y1).sum() / (g1y1.sum()+eps)
    fpr0 = (prob*g0y0).sum() / (g0y0.sum()+eps)
    fpr1 = (prob*g1y0).sum() / (g1y0.sum()+eps)
    return torch.max(torch.abs(tpr0-tpr1), torch.abs(fpr0-fpr1))

def _soft_dp(prob, sv_b, eps):
    sf  = sv_b.float()
    n0  = (1-sf).sum() + eps
    n1  = sf.sum()     + eps
    r0  = (prob * (1-sf)).sum() / n0
    r1  = (prob *    sf ).sum() / n1
    return torch.abs(r0 - r1)


def _build_disjoint_subgroups(y_true, y_prob, sv_list, attr_names, min_frac, max_frac):
    n      = len(y_true)
    min_sz = max(10, int(min_frac * n))
    max_sz = int(max_frac * n)
    valid_svs, valid_names = [], []
    for sv, nm in zip(sv_list, attr_names):
        sv_a = np.array(sv).astype(int)
        if len(np.unique(sv_a)) < 2: continue
        valid_svs.append(sv_a); valid_names.append(nm)
    n_attr = len(valid_svs)
    if n_attr == 0: return []
    subgroups = []; seen = set()
    for mask in range(1, 2**n_attr):
        active = [i for i in range(n_attr) if mask & (1 << i)]
        if len(active) > MAX_SG_ATTRS: continue
        for vals in range(2**len(active)):
            req = [(vals >> j) & 1 for j in range(len(active))]
            mem = np.ones(n, dtype=bool)
            for ai, rv in zip(active, req): mem &= (valid_svs[ai] == rv)
            key = mem.tobytes()
            if key in seen: continue
            seen.add(key); sz = mem.sum()
            if sz < min_sz or sz > max_sz: continue
            if len(np.unique(y_true[mem])) < 2: continue
            subgroups.append((mem, [valid_names[i] for i in active], req,
                              [(valid_names[i], req[j]) for j, i in enumerate(active)]))
    return subgroups

def compute_eo_subgroups_pairwise(y_true, y_prob, sv_list, attr_names, min_frac, max_frac):
    subgroups = _build_disjoint_subgroups(y_true, y_prob, sv_list, attr_names, min_frac, max_frac)
    n = len(y_true)
    if len(subgroups) < 2: return 0., None, np.zeros(n, dtype=int)
    worst = 0.; worst_grp = None; worst_mem = np.zeros(n, dtype=int)
    for i in range(len(subgroups)):
        memi, namei, reqi, _ = subgroups[i]
        for j in range(i+1, len(subgroups)):
            memj, _, _, _ = subgroups[j]
            if int((memi & memj).sum()) > 0: continue
            yi = y_true[memi]; pi = y_prob[memi]
            yj = y_true[memj]; pj = y_prob[memj]
            tpr_i = float(pi[yi==1].mean()) if (yi==1).sum() > 0 else 0.
            tpr_j = float(pj[yj==1].mean()) if (yj==1).sum() > 0 else 0.
            fpr_i = float(pi[yi==0].mean()) if (yi==0).sum() > 0 else 0.
            fpr_j = float(pj[yj==0].mean()) if (yj==0).sum() > 0 else 0.
            eo = max(abs(tpr_i-tpr_j), abs(fpr_i-fpr_j))
            if eo > worst:
                worst = eo
                worst_grp = {"attrs": namei, "vals": reqi, "n": int(memi.sum())}
                worst_mem = memi.astype(int)
    return float(worst), worst_grp, worst_mem

def compute_eo_subgroups(y_true, y_prob, sv_list, attr_names, min_frac, max_frac):
    n      = len(y_true)
    min_sz = max(10, int(min_frac * n))
    max_sz = int(max_frac * n)
    valid_svs, valid_names = [], []
    for sv, nm in zip(sv_list, attr_names):
        sv_a = np.array(sv).astype(int)
        if len(np.unique(sv_a)) < 2: continue
        valid_svs.append(sv_a); valid_names.append(nm)
    n_attr = len(valid_svs)
    if n_attr == 0: return 0., None, np.zeros(n, dtype=int)
    worst = 0.; worst_grp = None; worst_mem = np.zeros(n, dtype=int)
    seen = set()
    for mask in range(1, 2**n_attr):
        active = [i for i in range(n_attr) if mask & (1 << i)]
        if len(active) > MAX_SG_ATTRS: continue
        for vals in range(2**len(active)):
            req = [(vals >> j) & 1 for j in range(len(active))]
            mem = np.ones(n, dtype=bool)
            for ai, rv in zip(active, req): mem &= (valid_svs[ai] == rv)
            key = mem.tobytes()
            if key in seen: continue
            seen.add(key); sz = mem.sum()
            if sz < min_sz or sz > max_sz: continue
            sg_p = y_prob[mem];  sg_t = y_true[mem]
            ot_p = y_prob[~mem]; ot_t = y_true[~mem]
            if len(np.unique(sg_t)) < 2 or len(np.unique(ot_t)) < 2: continue
            tpr_sg  = float(sg_p[sg_t==1].mean()) if (sg_t==1).sum() > 0 else 0.
            tpr_oth = float(ot_p[ot_t==1].mean()) if (ot_t==1).sum() > 0 else 0.
            fpr_sg  = float(sg_p[sg_t==0].mean()) if (sg_t==0).sum() > 0 else 0.
            fpr_oth = float(ot_p[ot_t==0].mean()) if (ot_t==0).sum() > 0 else 0.
            eo = max(abs(tpr_sg-tpr_oth), abs(fpr_sg-fpr_oth))
            if eo > worst:
                worst = eo
                worst_grp = {"attrs": [valid_names[i] for i in active],
                             "vals": req, "n": int(sz)}
                worst_mem = mem.astype(int)
    return float(worst), worst_grp, worst_mem

def _make_membership(sv_bin_list, attr_names, active_attr_vals, n):
    mem = np.ones(n, dtype=bool)
    name_to_sv = {nm: sv for nm, sv in zip(attr_names, sv_bin_list)}
    for attr_nm, req_val in active_attr_vals:
        if attr_nm not in name_to_sv:
            return np.zeros(n, dtype=bool)
        mem &= (name_to_sv[attr_nm] == req_val)
    return mem

def eo_gap_auditor(y_val, p_val, sv_val_bin, attr_names, sv_tr_bin, min_frac, max_frac):
    subgroups = _build_disjoint_subgroups(y_val, p_val, sv_val_bin, attr_names, min_frac, max_frac)
    if len(subgroups) < 2: return []
    n_tr = len(sv_tr_bin[0])
    pairs = []
    for i in range(len(subgroups)):
        memi, namei, reqi, spec_i = subgroups[i]
        for j in range(i+1, len(subgroups)):
            memj, namej, reqj, spec_j = subgroups[j]
            if int((memi & memj).sum()) > 0: continue
            yi = y_val[memi]; pi = p_val[memi]
            yj = y_val[memj]; pj = p_val[memj]
            tpr_i = float(pi[yi==1].mean()) if (yi==1).sum() > 0 else 0.
            tpr_j = float(pj[yj==1].mean()) if (yj==1).sum() > 0 else 0.
            fpr_i = float(pi[yi==0].mean()) if (yi==0).sum() > 0 else 0.
            fpr_j = float(pj[yj==0].mean()) if (yj==0).sum() > 0 else 0.
            eo = max(abs(tpr_i-tpr_j), abs(fpr_i-fpr_j))
            if eo >= EO_GAP_FIRE_THRESH:
                mem_i_tr = _make_membership(sv_tr_bin, attr_names, spec_i, n_tr)
                mem_j_tr = _make_membership(sv_tr_bin, attr_names, spec_j, n_tr)
                pairs.append((eo, mem_i_tr.astype(np.float32), mem_j_tr.astype(np.float32),
                             namei, namej, spec_i, spec_j, int(memi.sum()), int(memj.sum()),
                             tpr_i, tpr_j, fpr_i, fpr_j))
    if not pairs: return []
    pairs.sort(key=lambda x: -x[0])
    flagged = []
    for rec in pairs[:TOP_K_INTERSECTIONS]:
        eo, memi_tr, memj_tr, namei, namej, spec_i, spec_j, ni, nj, tpr_i, tpr_j, fpr_i, fpr_j = rec
        spec_i_str = "+".join(f"{a}={v}" for a, v in spec_i) if spec_i else "?"
        spec_j_str = "+".join(f"{a}={v}" for a, v in spec_j) if spec_j else "?"
        tqdm.write(f"    │   EO-gap={eo:.4f}  [{spec_i_str}] (n={ni}, TPR={tpr_i:.3f}, FPR={fpr_i:.3f})"
                   f"  vs  [{spec_j_str}] (n={nj}, TPR={tpr_j:.3f}, FPR={fpr_j:.3f})")
        flagged.append({
            "memi_tr": memi_tr, "memj_tr": memj_tr, "eo_gap": float(eo),
            "spec_i": spec_i, "spec_j": spec_j, "n_i": ni, "n_j": nj,
            "tpr_i": tpr_i, "tpr_j": tpr_j, "fpr_i": fpr_i, "fpr_j": fpr_j,
        })
    return flagged


class GradientReversalFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha; return x.clone()
    @staticmethod
    def backward(ctx, grad):
        return -ctx.alpha * grad, None

class GRL(nn.Module):
    def __init__(self, alpha=1.0):
        super().__init__(); self.alpha = alpha
    def forward(self, x):
        return GradientReversalFn.apply(x, self.alpha)

class Encoder(nn.Module):
    def __init__(self, in_dim, hidden_dim, dropout):
        super().__init__()
        h = hidden_dim; self.rep_dim = h // 2 + 32
        self.net = nn.Sequential(
            nn.Linear(in_dim, h), nn.BatchNorm1d(h),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(h, self.rep_dim), nn.BatchNorm1d(self.rep_dim),
            nn.GELU(), nn.Dropout(dropout*0.5))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="linear")
                if m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self, x): return self.net(x)

class TaskHead(nn.Module):
    def __init__(self, rep_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(rep_dim, rep_dim//2), nn.GELU(),
            nn.Linear(rep_dim//2, 1))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, rep): return self.net(rep).squeeze(1)

class AdversaryHead(nn.Module):
    def __init__(self, rep_dim, hidden_dim, dropout):
        super().__init__()
        self.grl = GRL(alpha=1.0); h = hidden_dim // 2
        self.net = nn.Sequential(
            nn.Linear(rep_dim, h), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(h, h//2),   nn.GELU(), nn.Dropout(dropout*0.5),
            nn.Linear(h//2, 1))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight); nn.init.zeros_(m.bias)
    def set_alpha(self, a): self.grl.alpha = a
    def forward(self, z): return self.net(self.grl(z)).squeeze(1)

def _make_loader(tensors, batch_size, seed):
    ds = TensorDataset(*tensors)
    g  = torch.Generator(); g.manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=True,
                      drop_last=True, generator=g, num_workers=0)


def pretrain_encoder(X_tr, y_tr, wt, params, seed):
    set_seed(seed); p = params
    def _t(a): return torch.tensor(np.array(a), dtype=torch.float32).to(DEVICE)
    Xt = _t(X_tr); yt = _t(y_tr); wt_t = _t(wt)
    enc = Encoder(X_tr.shape[1], p["hidden_dim"], p["dropout"]).to(DEVICE)
    th  = TaskHead(enc.rep_dim).to(DEVICE)
    bce = nn.BCEWithLogitsLoss(reduction="none")
    opt = optim.AdamW(list(enc.parameters()) + list(th.parameters()),
                      lr=p["lr"], weight_decay=1e-4)
    loader = _make_loader([Xt, yt, wt_t], p["batch"], seed)
    enc.train(); th.train()
    for ep in range(p.get("pre_epochs", 8)):
        for xb, yb, wb in loader:
            opt.zero_grad(set_to_none=True)
            (bce(th(enc(xb)), yb) * wb).mean().backward()
            opt.step()
    tqdm.write(f"    │ pretrain: {p.get('pre_epochs', 8)} epochs done")
    return copy.deepcopy(enc.state_dict()), copy.deepcopy(th.state_dict()), enc.rep_dim


def _eval_and_log(proba, y_test, sv_te_bin, attr_names, min_frac, max_frac, tag="method"):
    pred          = (proba > 0.5).astype(int)
    per_eo, wc_eo = compute_eo_multi(y_test, proba, sv_te_bin, attr_names)
    per_dp, wc_dp = compute_dp_multi(proba, sv_te_bin, attr_names)
    sg_w, wg, _   = compute_eo_subgroups_pairwise(y_test, proba, sv_te_bin, attr_names, min_frac, max_frac)
    ece = compute_ece(y_test, proba)
    acc = float(accuracy_score(y_test, pred))
    auc = float(safe_auc(y_test, proba))
    eo_s = "PASS" if sg_w < TARGET_EO else "FAIL"
    dp_s = "PASS" if wc_dp < TARGET_DP else "FAIL"
    tqdm.write(f"    │ [{tag}]  WC-EO={wc_eo:.4f}  SG-EO={sg_w:.4f}({eo_s})"
               f"  WC-DP={wc_dp:.4f}({dp_s})  Acc={acc:.4f}  AUC={auc:.4f}"
               f"  ECE={ece:.4f}")
    if wg:
        attrs_str = "+".join(wg.get("attrs", []))
        tqdm.write(f"    │         worst subgroup pair attrs: [{attrs_str}]  n={wg.get('n','?')}")
    return {"eo": float(wc_eo), "sg_eo": float(sg_w), "acc": acc, "auc": auc,
            "dp": float(wc_dp), "ece": ece,
            "per_attr_eo": [float(x) for x in per_eo],
            "per_attr_dp": [float(x) for x in per_dp],
            "subgroup_worst": float(sg_w), "n_inter_heads": 0}


def run_vanilla_nn(X_tr, y_tr, wt, X_val, y_val, X_test, y_test,
                   sv_te_list, sv_val_list, attr_names,
                   enc_sd, th_sd, rep_dim, params, ds_cfg, seed):
    set_seed(seed); p = params
    min_frac = ds_cfg["min_frac"]; max_frac = ds_cfg["max_frac"]
    def _t(a): return torch.tensor(np.array(a), dtype=torch.float32).to(DEVICE)
    Xt = _t(X_tr); yt = _t(y_tr); wt_t = _t(wt)
    Xv = _t(X_val); Xte = _t(X_test)
    sv_te_bin  = [ensure_binary(sv) for sv in sv_te_list]
    sv_val_bin = [ensure_binary(sv) for sv in sv_val_list]
    enc = Encoder(X_tr.shape[1], p["hidden_dim"], p["dropout"]).to(DEVICE)
    th  = TaskHead(rep_dim).to(DEVICE)
    enc.load_state_dict(enc_sd); th.load_state_dict(th_sd)
    bce    = nn.BCEWithLogitsLoss(reduction="none")
    opt    = optim.AdamW(list(enc.parameters()) + list(th.parameters()),
                         lr=p["lr"], weight_decay=1e-4)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, p["epochs"], eta_min=p["lr"]*0.01)
    loader = _make_loader([Xt, yt, wt_t], p["batch"], seed)
    best_score = np.inf; best_enc = copy.deepcopy(enc.state_dict())
    best_th    = copy.deepcopy(th.state_dict()); no_imp = 0
    for epoch in range(p["epochs"]):
        enc.train(); th.train()
        for xb, yb, wb in loader:
            opt.zero_grad(set_to_none=True)
            (bce(th(enc(xb)), yb) * wb).mean().backward(); opt.step()
        sched.step()
        if epoch % p["eval_every"] == 0:
            enc.eval(); th.eval()
            with torch.no_grad():
                pv = torch.sigmoid(th(enc(Xv))).cpu().numpy()
            val_sg, _, _ = compute_eo_subgroups(y_val, pv, sv_val_bin, attr_names, min_frac, max_frac)
            val_auc = safe_auc(y_val, pv)
            score = val_sg + PARETO_LAMBDA*(1-val_auc) if not np.isnan(val_auc) else val_sg
            if score < best_score:
                best_score = score; best_enc = copy.deepcopy(enc.state_dict())
                best_th = copy.deepcopy(th.state_dict()); no_imp = 0
            else:
                no_imp += p["eval_every"]
            if no_imp >= p["patience"] * p["eval_every"]: break
    enc.load_state_dict(best_enc); th.load_state_dict(best_th)
    enc.eval(); th.eval()
    with torch.no_grad():
        proba = torch.sigmoid(th(enc(Xte))).cpu().numpy()
    return _eval_and_log(proba, y_test, sv_te_bin, attr_names, min_frac, max_frac, tag="vanilla-nn")


def run_zhang_adversarial(X_tr, y_tr, wt, sv_tr_list, sv_val_list, attr_names,
                          X_val, y_val, X_test, y_test, sv_te_list,
                          enc_sd, th_sd, rep_dim, params, ds_cfg, seed):
    set_seed(seed); p = params
    min_frac = ds_cfg["min_frac"]; max_frac = ds_cfg["max_frac"]
    def _t(a): return torch.tensor(np.array(a), dtype=torch.float32).to(DEVICE)
    sv_tr_bin  = [ensure_binary(sv) for sv in sv_tr_list]
    sv_te_bin  = [ensure_binary(sv) for sv in sv_te_list]
    sv_val_bin = [ensure_binary(sv) for sv in sv_val_list]
    n_base     = len(attr_names)
    Xt  = _t(X_tr); yt = _t(y_tr); wt_t = _t(wt)
    Xte = _t(X_test); Xv = _t(X_val)
    st_list = [_t(sv) for sv in sv_tr_bin]
    enc  = Encoder(X_tr.shape[1], p["hidden_dim"], p["dropout"]).to(DEVICE)
    th   = TaskHead(rep_dim).to(DEVICE)
    enc.load_state_dict(enc_sd); th.load_state_dict(th_sd)
    advs = nn.ModuleList([AdversaryHead(rep_dim, p["hidden_dim"], p["dropout"]).to(DEVICE) for _ in attr_names])
    bce     = nn.BCEWithLogitsLoss(reduction="none")
    opt_enc = optim.AdamW(list(enc.parameters()) + list(th.parameters()), lr=p["lr"], weight_decay=1e-4)
    opt_adv = optim.AdamW(advs.parameters(), lr=p["lr"] * p["adv_lr_mult"], weight_decay=1e-4)
    sched   = optim.lr_scheduler.CosineAnnealingLR(opt_enc, p["epochs"], eta_min=p["lr"]*0.01)
    loader  = _make_loader([Xt, yt, wt_t] + st_list, p["batch"], seed)
    best_score = np.inf; best_enc = copy.deepcopy(enc.state_dict())
    best_th_sd = copy.deepcopy(th.state_dict()); no_imp = 0
    for epoch in range(p["epochs"]):
        grl_alpha = p["grl_max"] * max(0.1, epoch / max(1, p["epochs"]))
        for h in advs: h.set_alpha(grl_alpha)
        enc.train(); th.train(); advs.train()
        for batch in loader:
            xb        = batch[0]; yb = batch[1]; wb = batch[2]
            sv_b_list = [batch[3+i] for i in range(n_base)]
            z_d    = enc(xb).detach()
            a_loss = sum(nn.functional.binary_cross_entropy_with_logits(
                advs[i](z_d), sv_b_list[i].float()) for i in range(n_base)) / n_base
            opt_adv.zero_grad(set_to_none=True); a_loss.backward()
            nn.utils.clip_grad_norm_(advs.parameters(), 1.0); opt_adv.step()
            opt_enc.zero_grad(set_to_none=True)
            z = enc(xb)
            task_loss = (bce(th(z), yb) * wb).mean()
            grl_loss  = sum(nn.functional.binary_cross_entropy_with_logits(
                advs[i](z), sv_b_list[i].float()) for i in range(n_base)) / n_base
            (p["task_lw"] * task_loss + grl_loss).backward()
            nn.utils.clip_grad_norm_(list(enc.parameters()) + list(th.parameters()), 0.5)
            opt_enc.step()
        sched.step()
        if epoch % p["eval_every"] == 0:
            enc.eval(); th.eval()
            with torch.no_grad():
                pv = torch.sigmoid(th(enc(Xv))).cpu().numpy()
            val_sg, _, _ = compute_eo_subgroups(y_val, pv, sv_val_bin, attr_names, min_frac, max_frac)
            val_auc = safe_auc(y_val, pv)
            val_acc = accuracy_score(y_val, (pv > 0.5).astype(int))
            score = val_sg + PARETO_LAMBDA*(1-val_auc) if not np.isnan(val_auc) else val_sg
            if val_acc >= p["min_acc_floor"] and score < best_score:
                best_score = score; best_enc = copy.deepcopy(enc.state_dict())
                best_th_sd = copy.deepcopy(th.state_dict()); no_imp = 0
            else:
                no_imp += p["eval_every"]
            if no_imp >= p["patience"] * p["eval_every"]: break
    enc.load_state_dict(best_enc); th.load_state_dict(best_th_sd)
    enc.eval(); th.eval()
    with torch.no_grad():
        proba = torch.sigmoid(th(enc(Xte))).cpu().numpy()
    return _eval_and_log(proba, y_test, sv_te_bin, attr_names, min_frac, max_frac, tag="zhang-adv")


def _group_encode(sv_bin_list):
    n = len(sv_bin_list[0]); g = np.zeros(n, dtype=int)
    for i, sv in enumerate(sv_bin_list):
        g += sv.astype(int) * (2**i)
    return g

def run_fairlearn_adv(X_tr, y_tr, X_test, y_test, sv_tr_list, sv_te_list, attr_names, params, ds_cfg, seed):
    if not HAS_FAIRLEARN:
        tqdm.write("    │ [fairlearn-adv] not installed")
        return None
    set_seed(seed); p = params
    min_frac = ds_cfg["min_frac"]; max_frac = ds_cfg["max_frac"]
    sv_tr_bin = [ensure_binary(sv) for sv in sv_tr_list]
    sv_te_bin = [ensure_binary(sv) for sv in sv_te_list]
    sf_tr = _group_encode(sv_tr_bin)
    n_groups = int(np.max(sf_tr)) + 1
    h = p["hidden_dim"]
    alpha = max(1.0, p.get("eo_coef", p.get("fairness_coef", 30.0)) / 15.0)
    fl_epochs = min(p["epochs"], 60)
    tqdm.write(f"    │ [fairlearn-adv] n_groups={n_groups}  alpha={alpha:.2f}  epochs={fl_epochs}"
               f" (capped from {p['epochs']} for wall-time budget)")
    try:
        clf = AdversarialFairnessClassifier(
            predictor_model=[h, "relu", h // 2, "relu"],
            adversary_model=[h // 4, "relu"],
            constraints="equalized_odds",
            learning_rate=p["lr"],
            alpha=alpha,
            epochs=fl_epochs,
            batch_size=p["batch"],
            shuffle=True,
            progress_updates=None,
            random_state=seed,
        )
        clf.fit(X_tr, y_tr, sensitive_features=sf_tr)
    except Exception as e:
        tqdm.write(f"    │ [fairlearn-adv] ERROR: {e}")
        return None
    try:
        proba = clf.predict_proba(X_test)[:, 1]
    except Exception:
        proba = clf.predict(X_test).astype(float)
    return _eval_and_log(proba, y_test, sv_te_bin, attr_names, min_frac, max_frac, tag="fairlearn-adv")


def _train_agad(X_tr, y_tr, wt, sv_tr_list, attr_names,
                X_val, y_val, sv_val_list,
                X_test, y_test, sv_te_list,
                enc_sd, th_sd, rep_dim,
                params, ds_cfg, verbose, seed,
                audit_trail=None):
    p        = params
    min_frac = ds_cfg["min_frac"]; max_frac = ds_cfg["max_frac"]
    eo_coef  = p["eo_coef"]; dp_coef = p["dp_coef"]
    task_lw  = p["task_lw"]; min_acc = p["min_acc_floor"]
    n_base   = len(sv_tr_list)
    set_seed(seed)
    eps = torch.tensor(1e-4, device=DEVICE)

    def _t(a, dtype=torch.float32):
        return torch.tensor(np.array(a), dtype=dtype).to(DEVICE)

    Xt  = _t(X_tr);  yt  = _t(y_tr);  wt_t = _t(wt)
    Xv  = _t(X_val); Xte = _t(X_test)
    sv_tr_bin  = [ensure_binary(sv) for sv in sv_tr_list]
    sv_te_bin  = [ensure_binary(sv) for sv in sv_te_list]
    sv_val_bin = [ensure_binary(sv) for sv in sv_val_list]
    st_list    = [_t(sv) for sv in sv_tr_bin]

    encoder   = Encoder(X_tr.shape[1], p["hidden_dim"], p["dropout"]).to(DEVICE)
    task_head = TaskHead(rep_dim).to(DEVICE)
    encoder.load_state_dict(enc_sd); task_head.load_state_dict(th_sd)
    bce = nn.BCEWithLogitsLoss(reduction="none")

    inter_adv_heads  = nn.ModuleList()
    inter_target_mem = []
    inter_meta       = []

    def _make_adv_opt():
        if len(inter_adv_heads) == 0: return None
        return optim.AdamW(inter_adv_heads.parameters(), lr=p["lr"] * p["adv_lr_mult"], weight_decay=1e-4)

    opt_enc = optim.AdamW(
        list(encoder.parameters()) + list(task_head.parameters()), lr=p["lr"], weight_decay=1e-4)
    opt_adv = None
    sched   = optim.lr_scheduler.CosineAnnealingLR(opt_enc, p["epochs"], eta_min=p["lr"]*0.01)

    idx_t  = torch.arange(len(y_tr), dtype=torch.long)
    loader = _make_loader([Xt, yt, wt_t, idx_t] + st_list, p["batch"], seed)

    best_score = np.inf; best_enc = copy.deepcopy(encoder.state_dict())
    best_th_sd = copy.deepcopy(task_head.state_dict())
    no_imp = 0; plateau_count = 0; best_sg_seen = np.inf; auditor_fires = 0
    epoch_times = []
    sg_eo_history = []
    fire_events = []

    pbar = tqdm(range(p["epochs"]), desc="    │ [agad]",
                dynamic_ncols=True, leave=False, disable=not verbose)

    for epoch in pbar:
        t_ep = time.time()
        grl_alpha = min(1.0, 0.1 + epoch / max(1, p["epochs"]))
        for h in inter_adv_heads: h.set_alpha(grl_alpha)
        encoder.train(); task_head.train()
        if inter_adv_heads: inter_adv_heads.train()

        for batch in loader:
            xb        = batch[0]; yb = batch[1]; wb = batch[2]
            bidx      = batch[3]
            sv_b_list = [batch[4+i] for i in range(n_base)]
            bidx_cpu  = bidx.cpu().numpy()

            if len(inter_adv_heads) > 0:
                for _ in range(p["adv_steps"]):
                    z_d = encoder(xb).detach()
                    a_loss = torch.tensor(0., device=DEVICE)
                    n_active = 0
                    for k, (mi_np, mj_np) in enumerate(inter_target_mem):
                        in_either = torch.tensor(
                            (mi_np[bidx_cpu] + mj_np[bidx_cpu]).clip(0, 1),
                            dtype=torch.bool, device=DEVICE)
                        if in_either.sum() < 4: continue
                        tgt = torch.tensor(mi_np[bidx_cpu], dtype=torch.float32, device=DEVICE)
                        a_loss = a_loss + nn.functional.binary_cross_entropy_with_logits(
                            inter_adv_heads[k].net(z_d).squeeze(1)[in_either], tgt[in_either])
                        n_active += 1
                    if n_active > 0:
                        opt_adv.zero_grad(set_to_none=True)
                        (a_loss / n_active).backward()
                        nn.utils.clip_grad_norm_(inter_adv_heads.parameters(), 1.0)
                        opt_adv.step()

            opt_enc.zero_grad(set_to_none=True)
            z = encoder(xb); logits = task_head(z); prob = torch.sigmoid(logits)
            task_loss = (bce(logits, yb) * wb).mean()
            eo_loss = sum(_soft_eo(prob, sv_b, yb.float(), eps) for sv_b in sv_b_list) / max(n_base, 1)
            dp_loss = sum(_soft_dp(prob, sv_b, eps) for sv_b in sv_b_list) / max(n_base, 1)

            inter_loss = torch.tensor(0., device=DEVICE)
            n_active_i = 0
            for k, (mi_np, mj_np) in enumerate(inter_target_mem):
                in_either = torch.tensor(
                    (mi_np[bidx_cpu] + mj_np[bidx_cpu]).clip(0, 1),
                    dtype=torch.bool, device=DEVICE)
                if in_either.sum() < 4: continue
                tgt = torch.tensor(mi_np[bidx_cpu], dtype=torch.float32, device=DEVICE)
                inter_loss = inter_loss + nn.functional.binary_cross_entropy_with_logits(
                    inter_adv_heads[k](z)[in_either], tgt[in_either])
                n_active_i += 1
            if n_active_i > 0:
                inter_loss = inter_loss / n_active_i

            loss = task_lw * task_loss + eo_coef * eo_loss + dp_coef * dp_loss + inter_loss
            loss.backward()
            nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(task_head.parameters()), 0.5)
            opt_enc.step()
        sched.step()
        epoch_times.append(time.time() - t_ep)

        if epoch % p["eval_every"] == 0:
            encoder.eval(); task_head.eval()
            with torch.no_grad():
                pv = torch.sigmoid(task_head(encoder(Xv))).cpu().numpy()
            val_acc = float(accuracy_score(y_val, (pv > 0.5).astype(int)))
            val_auc = float(safe_auc(y_val, pv))
            val_sg, _, _ = compute_eo_subgroups(y_val, pv, sv_val_bin, attr_names, min_frac, max_frac)
            _, val_dp_wc = compute_dp_multi(pv, sv_val_bin, attr_names)
            sg_eo_history.append((epoch, float(val_sg), float(val_dp_wc), float(val_acc)))

            if val_sg < best_sg_seen - 1e-4:
                best_sg_seen = val_sg; plateau_count = 0
            else:
                plateau_count += 1

            if plateau_count >= PLATEAU_PATIENCE:
                auditor_fires += 1
                tqdm.write(f"\n    │ [AUDITOR FIRE #{auditor_fires}] epoch={epoch}"
                           f"  val_sg_eo={val_sg:.4f} (plateaued {plateau_count} evals,"
                           f" best_seen={best_sg_seen:.4f})")
                plateau_count = 0
                flagged = eo_gap_auditor(
                    y_val, pv, sv_val_bin, attr_names, sv_tr_bin, min_frac, max_frac)
                fire_record = {"epoch": int(epoch), "val_sg_eo_at_fire": float(val_sg),
                               "val_dp_at_fire": float(val_dp_wc), "flagged_pairs": []}
                if flagged:
                    n_new = min(len(flagged), TOP_K_INTERSECTIONS)
                    inter_adv_heads  = nn.ModuleList()
                    inter_target_mem.clear()
                    inter_meta.clear()
                    for fi in range(n_new):
                        rec = flagged[fi]
                        inter_adv_heads.append(
                            AdversaryHead(rep_dim, p["hidden_dim"], p["dropout"]).to(DEVICE))
                        inter_target_mem.append((rec["memi_tr"], rec["memj_tr"]))
                        inter_meta.append(rec)
                        fire_record["flagged_pairs"].append({
                            "eo_gap": rec["eo_gap"], "spec_i": rec["spec_i"], "spec_j": rec["spec_j"],
                            "n_i": rec["n_i"], "n_j": rec["n_j"],
                            "tpr_i": rec["tpr_i"], "tpr_j": rec["tpr_j"],
                            "fpr_i": rec["fpr_i"], "fpr_j": rec["fpr_j"],
                        })
                    opt_adv = _make_adv_opt()
                    tqdm.write(f"    │ [AUDITOR] spawned {len(inter_adv_heads)} intersectional"
                               f" adversary heads targeting the pairs above\n")
                else:
                    tqdm.write(f"    │ [AUDITOR] scanned subgroup pairs, none exceeded"
                               f" gap threshold {EO_GAP_FIRE_THRESH} — no heads spawned\n")
                if audit_trail is not None:
                    audit_trail.append(fire_record)

            score = 2.0 * val_sg + 1.0 * val_dp_wc
            if not np.isnan(val_auc): score += PARETO_LAMBDA * (1 - val_auc)

            if val_acc >= min_acc and score < best_score:
                best_score = score; best_enc = copy.deepcopy(encoder.state_dict())
                best_th_sd = copy.deepcopy(task_head.state_dict()); no_imp = 0
            else:
                no_imp += p["eval_every"]

            if verbose:
                pbar.set_postfix({
                    "sg-EO": f"{val_sg:.4f}", "DP": f"{val_dp_wc:.4f}",
                    "acc": f"{val_acc:.3f}", "score": f"{score:.4f}",
                    "iheads": len(inter_adv_heads), "fires": auditor_fires})

            if no_imp >= p["patience"] * p["eval_every"]:
                tqdm.write(f"    │ [agad] early stop epoch={epoch}")
                break

    encoder.load_state_dict(best_enc); task_head.load_state_dict(best_th_sd)
    encoder.eval(); task_head.eval()
    with torch.no_grad():
        proba_te = torch.sigmoid(task_head(encoder(Xte))).cpu().numpy()
    res = _eval_and_log(proba_te, y_test, sv_te_bin, attr_names, min_frac, max_frac, tag="agad-full")
    res["n_inter_heads"]   = int(len(inter_adv_heads))
    res["auditor_fires"]   = int(auditor_fires)
    res["mean_epoch_sec"]  = float(np.mean(epoch_times)) if epoch_times else 0.
    res["total_train_sec"] = float(np.sum(epoch_times))
    res["sg_eo_history"]   = sg_eo_history

    if audit_trail is not None and len(audit_trail) > 0:
        first_fire_sg  = audit_trail[0]["val_sg_eo_at_fire"]
        post_fire_evals = [h for h in sg_eo_history if h[0] > audit_trail[0]["epoch"]]
        if post_fire_evals:
            min_after = min(h[1] for h in post_fire_evals)
            tqdm.write(f"    │ [AUDITOR MITIGATION] val SG-EO at first fire={first_fire_sg:.4f}"
                       f" -> best val SG-EO afterward={min_after:.4f}"
                       f" (Δ={min_after-first_fire_sg:+.4f})")

    return res


def _cache_path(name):
    return f"/tmp/agad{CACHE_VERSION}_{name}.pkl"

def preprocess_acs_employment(use_cache=True):
    cache = _cache_path("acs_employment")
    if use_cache and os.path.exists(cache): return joblib.load(cache)
    if not HAS_FOLKTABLES: raise RuntimeError("pip install folktables")
    data_source = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
    acs_data    = data_source.get_data(states=ACS_STATES, download=True)
    features, label, group = ACSEmployment.df_to_numpy(acs_data)
    valid    = ~np.isnan(label.astype(float))
    features = features[valid].astype(np.float32); label = label[valid].astype(int)
    acs_elig = acs_data[valid].reset_index(drop=True)
    RACE_MAP_4  = {1: 0, 2: 1, 3: 3, 4: 2, 5: 2, 6: 3, 7: 3, 8: 3, 9: 3}
    RACE_NAMES4 = ["White", "Black", "Asian_PI", "Other"]
    rac1p = acs_elig["RAC1P"].values; hisp = acs_elig["HISP"].values
    race  = np.array([RACE_MAP_4.get(int(r), 3) for r in rac1p])
    race[hisp != 1] = 3
    sex = (acs_elig["SEX"].values == 2).astype(int)
    tr, te = stratified_split(features, label, [race, sex])
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(features[tr]); X_te_s = scaler.transform(features[te])
    res = {"X_train_nn": X_tr_s, "X_test_nn": X_te_s,
           "y_train": label[tr], "y_test": label[te]}
    for g, nm in enumerate(RACE_NAMES4):
        arr_tr = (race[tr] == g).astype(int); arr_te = (race[te] == g).astype(int)
        if arr_te.mean() >= 0.02:
            res[f"race_{nm}_train"] = arr_tr; res[f"race_{nm}_test"] = arr_te
        else:
            tqdm.write(f"  [exclude] race_{nm} = {arr_te.mean():.3f} — too small")
    res["sex_Female_train"] = sex[tr]; res["sex_Female_test"] = sex[te]
    DATASET_CONFIG["acs_employment"]["sens_attrs"] = [
        k.replace("_train", "") for k in res
        if k.endswith("_train") and k not in ("y_train", "X_train_nn")]
    joblib.dump(res, cache); return res

def preprocess_adult(use_cache=True):
    cache = _cache_path("adult")
    if use_cache and os.path.exists(cache): return joblib.load(cache)
    from ucimlrepo import fetch_ucirepo
    repo  = fetch_ucirepo(id=2)
    X_df  = repo.data.features.copy(); y_ser = repo.data.targets.copy()
    y_raw      = y_ser.iloc[:, 0].astype(str).str.strip().str.rstrip(".")
    y          = (y_raw == ">50K").astype(int).values
    race_raw   = X_df["race"].astype(str).str.strip()
    race_Black = (race_raw == "Black").astype(int).values
    race_White = (race_raw == "White").astype(int).values
    sex        = (X_df["sex"].astype(str).str.strip() == "Female").astype(int).values
    drop_cols  = ["race", "sex", "fnlwgt", "education-num"]
    X_df = X_df.drop(columns=[c for c in drop_cols if c in X_df.columns])
    X_df = pd.get_dummies(X_df); X = X_df.values.astype(np.float32)
    valid = (~np.isnan(X).any(axis=1)) & (~pd.isna(y_raw).values)
    X, y = X[valid], y[valid]
    race_Black = race_Black[valid]; race_White = race_White[valid]; sex = sex[valid]
    tr, te = stratified_split(X, y, [race_Black, sex])
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X[tr]); X_te_s = scaler.transform(X[te])
    res = {"X_train_nn": X_tr_s, "X_test_nn": X_te_s,
           "y_train": y[tr], "y_test": y[te],
           "race_Black_train": race_Black[tr], "race_Black_test": race_Black[te],
           "race_White_train": race_White[tr], "race_White_test": race_White[te],
           "sex_Female_train": sex[tr],        "sex_Female_test": sex[te]}
    joblib.dump(res, cache); return res

def preprocess_communities_crime(use_cache=True):
    cache = _cache_path("communities_crime")
    if use_cache and os.path.exists(cache): return joblib.load(cache)
    from ucimlrepo import fetch_ucirepo
    repo  = fetch_ucirepo(id=183)
    X_df  = repo.data.features.copy(); y_df = repo.data.targets.copy()
    y_cont = pd.to_numeric(y_df.iloc[:, 0], errors="coerce").values
    valid  = ~np.isnan(y_cont); y_cont = y_cont[valid]
    X_df   = X_df[valid].reset_index(drop=True)
    y      = (y_cont > np.median(y_cont)).astype(int)
    def find_col(df, patterns):
        for pat in patterns:
            matches = [c for c in df.columns if pat.lower() in c.lower()]
            if matches: return pd.to_numeric(df[matches[0]], errors="coerce")
        return None
    col_black  = find_col(X_df, ["racepctblack", "pctblack"])
    col_income = find_col(X_df, ["medIncome", "medincome"])
    col_unemp  = find_col(X_df, ["PctUnemployed", "pctunemployed"])
    def binarise(col, high_is_1=True):
        if col is None: return None
        col = col.fillna(col.median()).values; med = np.median(col)
        return (col > med).astype(int) if high_is_1 else (col < med).astype(int)
    s_black  = binarise(col_black,  high_is_1=True)
    s_income = binarise(col_income, high_is_1=False)
    s_unemp  = binarise(col_unemp,  high_is_1=True)
    n = len(y)
    if s_black  is None: tqdm.write("  [warn] racepctblack not found"); s_black  = np.zeros(n, dtype=int)
    if s_income is None: tqdm.write("  [warn] medIncome not found");    s_income = np.zeros(n, dtype=int)
    if s_unemp  is None: tqdm.write("  [warn] PctUnemployed not found");s_unemp  = np.zeros(n, dtype=int)
    X_df_num  = X_df.apply(pd.to_numeric, errors="coerce")
    X_df_num  = X_df_num.dropna(axis=1, thresh=int(0.7*len(X_df_num)))
    X_df_num  = X_df_num.fillna(X_df_num.median())
    sens_drop = ["racepct", "racePct", "medIncome", "PctUnemployed", "pctFemale", "ViolentCrimesPerPop"]
    drop_cols = [c for c in X_df_num.columns
                 if any(pt.lower() in c.lower() for pt in sens_drop)]
    X = X_df_num.drop(columns=drop_cols, errors="ignore").values.astype(np.float32)
    tr, te = stratified_split(X, y, [s_black, s_income, s_unemp])
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X[tr]); X_te_s = scaler.transform(X[te])
    res = {"X_train_nn": X_tr_s, "X_test_nn": X_te_s,
           "y_train": y[tr], "y_test": y[te],
           "high_black_pct_train":    s_black[tr],  "high_black_pct_test":    s_black[te],
           "low_income_train":        s_income[tr], "low_income_test":        s_income[te],
           "high_unemployment_train": s_unemp[tr],  "high_unemployment_test": s_unemp[te]}
    joblib.dump(res, cache); return res

PREPROCESS_FNS = {
    "acs_employment":    preprocess_acs_employment,
    "adult":             preprocess_adult,
    "communities_crime": preprocess_communities_crime,
}


def run_optuna_for_dataset(ds_key, data):
    if not HAS_OPTUNA:
        tqdm.write("  [optuna] not installed — using base params")
        return BASE_PARAMS[ds_key], None

    tqdm.write(_banner(f"OPTUNA TUNING: {ds_key.upper()}", width=80))

    cfg        = DATASET_CONFIG[ds_key]
    base_p     = BASE_PARAMS[ds_key]
    attr_names = cfg["sens_attrs"]

    X_train = to_dense(data["X_train_nn"])
    y_train = np.array(data["y_train"])
    sv_tr_full = [np.array(data[f"{a}_train"]) for a in attr_names]

    seed = OPTUNA_SEED_SINGLE
    tr_idx, val_idx = build_val_split(y_train, sv_tr_full, frac=0.15, seed=seed)
    X_tr   = X_train[tr_idx]; X_val  = X_train[val_idx]
    y_tr   = y_train[tr_idx]; y_val  = y_train[val_idx]
    sv_tr  = [sv[tr_idx]  for sv in sv_tr_full]
    sv_val = [sv[val_idx] for sv in sv_tr_full]

    if len(y_tr) > OPTUNA_MAX_TRAIN_ROWS:
        rng = np.random.RandomState(seed)
        sub_idx = rng.choice(len(y_tr), size=OPTUNA_MAX_TRAIN_ROWS, replace=False)
        X_tr  = X_tr[sub_idx]; y_tr = y_tr[sub_idx]
        sv_tr = [sv[sub_idx] for sv in sv_tr]
        tqdm.write(f"  [optuna] {ds_key}: subsampled train set to {OPTUNA_MAX_TRAIN_ROWS}/"
                   f"{len(tr_idx)} rows for tuning speed (final 3-seed run uses full data)")

    wt = compute_label_reweights(y_tr, sv_tr)

    tune_epochs    = max(12, int(base_p["epochs"] * OPTUNA_EPOCHS_FRAC))
    hard_acc_floor = base_p["min_acc_floor"]

    trial_log = []

    def objective(trial):
        p = dict(base_p)
        p["eo_coef"]    = trial.suggest_float("eo_coef",    2.0,  60.0, log=True)
        p["dp_coef"]    = trial.suggest_float("dp_coef",    1.0,  40.0, log=True)
        p["adv_lr_mult"]= trial.suggest_float("adv_lr_mult", 1.5, 10.0, log=True)
        p["adv_steps"]  = trial.suggest_int  ("adv_steps",   1,   6)
        p["task_lw"]    = trial.suggest_float("task_lw",     1.5, 6.0)
        p["dropout"]    = trial.suggest_float("dropout",     0.05,0.35)
        p["lr"]         = trial.suggest_float("lr",          1e-4,2e-3, log=True)
        p["epochs"]        = tune_epochs
        p["patience"]      = max(8, base_p["patience"])
        p["min_acc_floor"] = hard_acc_floor

        set_seed(seed)
        try:
            enc_sd, th_sd, rep_dim = pretrain_encoder(X_tr, y_tr, wt, p, seed)
            res = _train_agad(
                X_tr, y_tr, wt, sv_tr, attr_names,
                X_val, y_val, sv_val,
                X_val, y_val, sv_val,
                enc_sd, th_sd, rep_dim,
                p, cfg, verbose=False, seed=seed, audit_trail=None)
        except Exception as e:
            tqdm.write(f"  [optuna] trial failed: {e}")
            return 900.0

        sg_eo   = res.get("sg_eo", 1.0)
        dp      = res.get("dp", 1.0)
        acc     = res.get("acc", 0.0)
        auc     = res.get("auc", 0.0)
        fires   = res.get("auditor_fires", 0)
        n_heads = res.get("n_inter_heads", 0)

        trial_log.append({"trial": trial.number, "sg_eo": sg_eo, "dp": dp, "acc": acc,
                          "auc": auc, "fires": fires, "n_heads": n_heads,
                          "params": {k: p[k] for k in ["eo_coef","dp_coef","adv_lr_mult",
                                                        "adv_steps","task_lw","dropout","lr"]}})

        if acc < hard_acc_floor:
            return 800.0 + (hard_acc_floor - acc) * 10.0

        auditor_bonus = 0.003 * fires + 0.001 * n_heads
        score = sg_eo + 0.5 * dp + 0.8 * (1.0 - acc) + PARETO_LAMBDA * (1.0 - auc) - auditor_bonus
        return float(score)

    sampler = optuna.samplers.TPESampler(seed=SEED, n_startup_trials=10)
    study   = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=OPTUNA_N_TRIALS_PER_DS,
                   show_progress_bar=False, gc_after_trial=True)

    best  = study.best_params
    tuned = dict(base_p)
    tuned.update(best)
    tuned["epochs"]        = base_p["epochs"]
    tuned["min_acc_floor"] = hard_acc_floor

    tqdm.write(f"\n  [OPTUNA RESULT] {ds_key}  best trial #{study.best_trial.number}"
               f"  score={study.best_value:.4f}")
    tqdm.write(f"  [OPTUNA RESULT] best hyperparameters:")
    for k, v in best.items():
        vs = f"{v:.5f}" if isinstance(v, float) else str(v)
        tqdm.write(f"      {k:<16} = {vs}")
    feasible = [t for t in trial_log if t["acc"] >= hard_acc_floor]
    best_trial_sg_eo = float("nan")
    if 0 <= study.best_trial.number < len(trial_log):
        best_trial_sg_eo = trial_log[study.best_trial.number]["sg_eo"]
    if feasible:
        tqdm.write(f"  [OPTUNA RESULT] {len(feasible)}/{len(trial_log)} trials met the"
                   f" accuracy floor ({hard_acc_floor:.2f}); best trial sg_eo={best_trial_sg_eo:.4f}")
    else:
        tqdm.write(f"  [OPTUNA RESULT] WARNING: 0/{len(trial_log)} trials met the accuracy floor"
                   f" ({hard_acc_floor:.2f}) — fairness/accuracy tradeoff may be infeasible at this"
                   f" floor; consider lowering min_acc_floor for this dataset")
    return tuned, trial_log


def _run_dataset(data, ds_key, params, zhang_params):
    cfg      = DATASET_CONFIG[ds_key]
    X_train    = to_dense(data["X_train_nn"])
    X_test     = to_dense(data["X_test_nn"])
    y_train    = np.array(data["y_train"])
    y_test     = np.array(data["y_test"])
    attr_names = cfg["sens_attrs"]
    sv_tr_full = [np.array(data[f"{a}_train"]) for a in attr_names]
    sv_te      = [np.array(data[f"{a}_test"])  for a in attr_names]

    metric_keys = ["eo", "sg_eo", "acc", "auc", "dp", "ece",
                   "per_attr_eo", "per_attr_dp", "subgroup_worst",
                   "n_inter_heads", "auditor_fires",
                   "mean_epoch_sec", "total_train_sec"]
    results = {m: {k: [] for k in metric_keys} for m in METHODS}
    audit_trails_per_seed = []

    for si, seed in enumerate(SEEDS):
        tqdm.write(_banner(f"Seed {si+1}/{N_SEEDS}  (seed={seed})", width=72, char="─"))
        tr_idx, val_idx = build_val_split(y_train, sv_tr_full, frac=0.15, seed=seed)
        X_tr   = X_train[tr_idx];  X_val  = X_train[val_idx]
        y_tr   = y_train[tr_idx];  y_val  = y_train[val_idx]
        sv_tr  = [sv[tr_idx]  for sv in sv_tr_full]
        sv_val = [sv[val_idx] for sv in sv_tr_full]
        tqdm.write(f"    │ split: train={len(y_tr)}  val={len(y_val)}  test={len(y_test)}"
                   f"  label_rate={y_tr.mean():.3f}")
        wt = compute_label_reweights(y_tr, sv_tr)
        tqdm.write(f"    │ reweights: min={wt.min():.3f}  max={wt.max():.3f}  mean={wt.mean():.3f}")

        enc_sd_v, th_sd_v, rep_dim_v = pretrain_encoder(X_tr, y_tr, wt, params, seed)
        tqdm.write(f"    ├─ [1/{len(METHODS)}] Vanilla NN")
        r = run_vanilla_nn(X_tr, y_tr, wt, X_val, y_val, X_test, y_test,
                           sv_te, sv_val, attr_names, enc_sd_v, th_sd_v, rep_dim_v, params, cfg, seed)
        for k in metric_keys:
            if k in r: results["vanilla_nn"][k].append(r[k])

        enc_sd_z, th_sd_z, rep_dim_z = pretrain_encoder(X_tr, y_tr, wt, zhang_params, seed)
        tqdm.write(f"    ├─ [2/{len(METHODS)}] Zhang (GRL-only)")
        r = run_zhang_adversarial(X_tr, y_tr, wt, sv_tr, sv_val, attr_names,
                                  X_val, y_val, X_test, y_test, sv_te,
                                  enc_sd_z, th_sd_z, rep_dim_z, zhang_params, cfg, seed)
        for k in metric_keys:
            if k in r: results["zhang_adversarial"][k].append(r[k])

        mi = 3
        if HAS_FAIRLEARN:
            tqdm.write(f"    ├─ [{mi}/{len(METHODS)}] FairLearn Adversarial")
            r = run_fairlearn_adv(X_tr, y_tr, X_test, y_test, sv_tr, sv_te, attr_names, params, cfg, seed)
            if r is not None:
                for k in metric_keys:
                    if k in r: results["fairlearn_adv"][k].append(r[k])
            mi += 1

        enc_sd_a, th_sd_a, rep_dim_a = pretrain_encoder(X_tr, y_tr, wt, params, seed)
        tqdm.write(f"    └─ [{mi}/{len(METHODS)}] AGAD Full")
        audit_trail = []
        r = _train_agad(X_tr, y_tr, wt, sv_tr, attr_names,
                        X_val, y_val, sv_val, X_test, y_test, sv_te,
                        enc_sd_a, th_sd_a, rep_dim_a, params, cfg, verbose=True, seed=seed,
                        audit_trail=audit_trail)
        for k in metric_keys:
            if k in r: results["agad_full"][k].append(r[k])
        audit_trails_per_seed.append({"seed": seed, "fires": audit_trail})

        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    return results, audit_trails_per_seed


def run_tradeoff_diagnostic(ds_key, data, base_params):
    tqdm.write(_banner(f"FAIRNESS-ACCURACY TRADEOFF DIAGNOSTIC: {ds_key.upper()}", width=80))
    cfg        = DATASET_CONFIG[ds_key]
    attr_names = cfg["sens_attrs"]
    X_train    = to_dense(data["X_train_nn"])
    y_train    = np.array(data["y_train"])
    sv_tr_full = [np.array(data[f"{a}_train"]) for a in attr_names]

    seed = OPTUNA_SEED_SINGLE
    tr_idx, val_idx = build_val_split(y_train, sv_tr_full, frac=0.15, seed=seed)
    X_tr   = X_train[tr_idx]; X_val  = X_train[val_idx]
    y_tr   = y_train[tr_idx]; y_val  = y_train[val_idx]
    sv_tr  = [sv[tr_idx]  for sv in sv_tr_full]
    sv_val = [sv[val_idx] for sv in sv_tr_full]

    if len(y_tr) > OPTUNA_MAX_TRAIN_ROWS:
        rng = np.random.RandomState(seed)
        sub_idx = rng.choice(len(y_tr), size=OPTUNA_MAX_TRAIN_ROWS, replace=False)
        X_tr  = X_tr[sub_idx]; y_tr = y_tr[sub_idx]
        sv_tr = [sv[sub_idx] for sv in sv_tr]
        tqdm.write(f"  [diagnostic] {ds_key}: subsampled train set to {OPTUNA_MAX_TRAIN_ROWS}/"
                   f"{len(tr_idx)} rows for sweep speed")

    wt = compute_label_reweights(y_tr, sv_tr)

    diag_epochs = max(10, int(base_params["epochs"] * 0.40))
    base_eo = base_params["eo_coef"]
    base_dp = base_params["dp_coef"]
    multipliers = np.geomspace(0.1, 4.0, DIAG_SWEEP_POINTS)

    rows = []
    tqdm.write(f"  Sweeping eo_coef/dp_coef jointly across {DIAG_SWEEP_POINTS} multipliers"
               f" of tuned values (eo_coef={base_eo:.2f}, dp_coef={base_dp:.2f}),"
               f" {diag_epochs} epochs each, single seed={seed}")
    tqdm.write(f"  {'mult':>6} {'eo_coef':>9} {'dp_coef':>9} {'val_SG-EO':>10} {'val_DP':>9} {'val_Acc':>9}")
    for mult in multipliers:
        p = dict(base_params)
        p["eo_coef"] = base_eo * mult
        p["dp_coef"] = base_dp * mult
        p["epochs"]  = diag_epochs
        p["min_acc_floor"] = 0.0
        set_seed(seed)
        enc_sd, th_sd, rep_dim = pretrain_encoder(X_tr, y_tr, wt, p, seed)
        res = _train_agad(X_tr, y_tr, wt, sv_tr, attr_names,
                          X_val, y_val, sv_val, X_val, y_val, sv_val,
                          enc_sd, th_sd, rep_dim, p, cfg, verbose=False, seed=seed,
                          audit_trail=None)
        rows.append({"mult": float(mult), "eo_coef": p["eo_coef"], "dp_coef": p["dp_coef"],
                     "sg_eo": res["sg_eo"], "dp": res["dp"], "acc": res["acc"], "auc": res["auc"]})
        tqdm.write(f"  {mult:6.2f} {p['eo_coef']:9.2f} {p['dp_coef']:9.2f}"
                   f" {res['sg_eo']:10.4f} {res['dp']:9.4f} {res['acc']:9.4f}")
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    accs   = [r["acc"]   for r in rows]
    sg_eos = [r["sg_eo"] for r in rows]
    if len(accs) >= 2 and np.std(sg_eos) > 1e-9:
        corr = float(np.corrcoef(sg_eos, accs)[0, 1])
    else:
        corr = float("nan")
    tqdm.write(f"\n  [DIAGNOSTIC SUMMARY] {ds_key}: correlation(SG-EO, Accuracy) across sweep = {corr:.3f}"
               f"  (more negative => stronger fairness/accuracy tradeoff)")
    acc_drop = max(accs) - min(accs)
    eo_drop  = max(sg_eos) - min(sg_eos)
    tqdm.write(f"  [DIAGNOSTIC SUMMARY] accuracy range across sweep = {acc_drop:.4f}"
               f"   SG-EO range across sweep = {eo_drop:.4f}")
    return rows


def _stats(mr, m, key):
    vals = [_to_float(v) for v in mr.get(m, {}).get(key, [])]
    vals = [v for v in vals if v is not None]
    if not vals: return float("nan"), float("nan")
    return float(np.mean(vals)), float(np.std(vals))


def print_tuned_params_summary(tuned_params):
    print(f"\n{'═'*100}")
    print("  BEST HYPERPARAMETERS FOUND BY OPTUNA (per dataset)")
    print(f"{'─'*100}")
    for ds_key, p in tuned_params.items():
        print(f"\n  {DS_LABELS.get(ds_key, ds_key)}:")
        keys_to_show = ["eo_coef", "dp_coef", "adv_lr_mult", "adv_steps", "task_lw",
                        "dropout", "lr", "min_acc_floor", "epochs", "hidden_dim", "batch"]
        for k in keys_to_show:
            if k in p:
                v = p[k]
                vs = f"{v:.5f}" if isinstance(v, float) else str(v)
                print(f"      {k:<16} = {vs}")
    print(f"{'═'*100}\n")


def print_auditor_report(all_audit_trails):
    print(f"\n{'═'*120}")
    print("  AUDITOR ACTIVITY REPORT — what it found and how it was mitigated")
    print(f"{'─'*120}")
    for ds_key, seed_trails in all_audit_trails.items():
        print(f"\n  {'▶ ' + DS_LABELS.get(ds_key, ds_key).upper()}")
        for st in seed_trails:
            seed = st["seed"]; fires = st["fires"]
            if not fires:
                print(f"    seed={seed}: auditor never fired (SG-EO never plateaued above"
                      f" threshold during training)")
                continue
            print(f"    seed={seed}: {len(fires)} fire event(s)")
            for fi, fr in enumerate(fires):
                print(f"      fire #{fi+1} @ epoch {fr['epoch']}:"
                      f" val SG-EO={fr['val_sg_eo_at_fire']:.4f}  val DP={fr['val_dp_at_fire']:.4f}")
                if not fr["flagged_pairs"]:
                    print(f"        -> no subgroup pair exceeded gap threshold"
                          f" {EO_GAP_FIRE_THRESH}; no heads spawned")
                    continue
                for pi, pair in enumerate(fr["flagged_pairs"]):
                    si = "+".join(f"{a}={v}" for a, v in pair["spec_i"]) if pair["spec_i"] else "?"
                    sj = "+".join(f"{a}={v}" for a, v in pair["spec_j"]) if pair["spec_j"] else "?"
                    print(f"        pair {pi+1}: EO-gap={pair['eo_gap']:.4f}"
                          f"  [{si}] (n={pair['n_i']}, TPR={pair['tpr_i']:.3f}, FPR={pair['fpr_i']:.3f})"
                          f"  vs  [{sj}] (n={pair['n_j']}, TPR={pair['tpr_j']:.3f}, FPR={pair['fpr_j']:.3f})")
                    print(f"        -> mitigation: spawned a dedicated adversarial head trained"
                          f" to predict membership in [{si}] vs [{sj}] from the representation;"
                          f" the encoder is then pushed (via this head's reversed gradient) to make"
                          f" that specific pair indistinguishable, directly closing this intersectional gap.")
    print(f"{'═'*120}\n")


def print_results(all_results):
    W = 130
    print(f"\n{'═'*W}")
    print(f"  COMPARISON: Vanilla NN vs Zhang(GRL-only) vs FairLearn vs AGAD(full)")
    print(f"  AGAD loss = task_loss + eo_coef*soft_EO + dp_coef*soft_DP + auditor_intersectional_adversary")
    print(f"  (base GRL adversary removed from AGAD — redundant with direct soft-EO/DP; Zhang baseline keeps it)")
    print(f"  Targets: SG-EO<{TARGET_EO}  DP<{TARGET_DP}  |  seeds={SEEDS}  N={N_SEEDS}  device={DEVICE}")
    print(f"{'─'*W}")

    for ds_key, mr in all_results.items():
        print(f"\n  {'▶ ' + DS_LABELS.get(ds_key, ds_key).upper()}")
        hdr = (f"  {'Method':<24} {'WC-EO':>8} {'SG-EO':>10} {'SG-EO±':>8} {'WC-DP':>8}"
               f" {'Acc':>8} {'Acc±':>7} {'AUC':>8} {'ECE':>7}  EO   DP  Extra")
        print(hdr); print(f"  {'─'*W}")
        for m in METHODS:
            if m not in mr or not mr[m].get("sg_eo"): continue
            eo_m, _    = _stats(mr, m, "eo")
            sg_m, sg_s = _stats(mr, m, "sg_eo")
            dp_m, _    = _stats(mr, m, "dp")
            ac_m, ac_s = _stats(mr, m, "acc")
            au_m, _    = _stats(mr, m, "auc")
            ec_m, _    = _stats(mr, m, "ece")
            n_h = int(np.nanmean([v for v in mr[m].get("n_inter_heads", []) if v is not None] or [0]))
            af  = int(np.nanmean([v for v in mr[m].get("auditor_fires",  []) if v is not None] or [0]))
            def _f(v): return f"{v:8.4f}" if not np.isnan(v) else "     nan"
            def _fs(v): return f"{v:7.4f}" if not np.isnan(v) else "    nan"
            s_eo = "PASS" if (not np.isnan(sg_m) and sg_m < TARGET_EO) else "FAIL"
            s_dp = "PASS" if (not np.isnan(dp_m) and dp_m < TARGET_DP)  else "FAIL"
            label = METHOD_LABELS.get(m, m)
            extra = f"  iheads={n_h}  fires={af}" if m == "agad_full" else ""
            print(f"  {label:<24} {_f(eo_m)} {_f(sg_m)} {_fs(sg_s)} {_f(dp_m)}"
                  f" {_f(ac_m)} {_fs(ac_s)} {_f(au_m)} {_f(ec_m)}  {s_eo}  {s_dp}{extra}")

        for baseline in ["zhang_adversarial", "fairlearn_adv"]:
            if baseline in mr and mr[baseline].get("sg_eo") and "agad_full" in mr:
                sg_b, _ = _stats(mr, baseline,   "sg_eo")
                sg_a, _ = _stats(mr, "agad_full", "sg_eo")
                dp_b, _ = _stats(mr, baseline,   "dp")
                dp_a, _ = _stats(mr, "agad_full", "dp")
                ac_b, _ = _stats(mr, baseline,   "acc")
                ac_a, _ = _stats(mr, "agad_full", "acc")
                if not np.isnan(sg_b) and not np.isnan(sg_a):
                    tag = "improved" if sg_b - sg_a > 0.002 else ("marginal" if sg_b > sg_a else "flat/worse")
                    bl_label = METHOD_LABELS.get(baseline, baseline)
                    print(f"    vs {bl_label}: SG-EO {sg_b:.4f}->{sg_a:.4f} (Δ={sg_a-sg_b:+.4f} {tag})"
                          f"   DP {dp_b:.4f}->{dp_a:.4f}   Acc {ac_b:.4f}->{ac_a:.4f}"
                          f" (Δacc={ac_a-ac_b:+.4f})")

    print(f"\n{'═'*W}")
    print(f"  PASS/FAIL MATRIX  (mean over {N_SEEDS} seeds, EO<{TARGET_EO}  DP<{TARGET_DP})\n")
    hdr2 = f"  {'Dataset':<26}" + "".join(f" {METHOD_LABELS.get(m,'?'):>30}" for m in METHODS)
    print(hdr2); print(f"  {'─'*W}")
    for ds_key, mr in all_results.items():
        row = f"  {DS_LABELS.get(ds_key, ds_key):<26}"
        for m in METHODS:
            sg, sg_s = _stats(mr, m, "sg_eo"); dp, _ = _stats(mr, m, "dp")
            ac, _    = _stats(mr, m, "acc")
            if np.isnan(sg): cell = "           N/A          "
            else:
                eo_p = "EO+" if sg < TARGET_EO else "EO-"
                dp_p = "DP+" if dp < TARGET_DP else "DP-"
                cell = f"{eo_p} {dp_p} EO={sg:.3f} Acc={ac:.3f}"
            row += f" {cell:>30}"
        print(row)

    print(f"\n{'─'*W}")
    print(f"  COMPUTE (AGAD only, mean over seeds)\n")
    print(f"  {'Dataset':<26} {'mean_epoch_sec':>16} {'total_train_sec':>17}")
    for ds_key, mr in all_results.items():
        me, _ = _stats(mr, "agad_full", "mean_epoch_sec")
        tt, _ = _stats(mr, "agad_full", "total_train_sec")
        def _fc(v): return f"{v:>16.1f}" if not np.isnan(v) else "             nan"
        print(f"  {DS_LABELS.get(ds_key, ds_key):<26} {_fc(me)} {_fc(tt)}")
    print(f"{'═'*W}\n")


def plot_results(all_results, save_dir="/kaggle/working", show=True):
    if not HAS_MPL:
        tqdm.write("  [plot] matplotlib not available — skipping")
        return []

    os.makedirs(save_dir, exist_ok=True)
    saved = []
    active_methods = [m for m in METHODS if any(
        mr.get(m, {}).get("sg_eo") for mr in all_results.values())]
    n_m  = len(active_methods)
    n_ds = len(all_results)

    def _mean_std(mr, m, key):
        vals = [_to_float(v) for v in mr.get(m, {}).get(key, [])]
        vals = [v for v in vals if v is not None]
        if not vals: return float("nan"), 0.
        return float(np.mean(vals)), float(np.std(vals))

    metrics_cfg = [
        ("sg_eo", "SG-EO (lower=fairer)", TARGET_EO, True),
        ("dp",    "WC-DP (lower=fairer)", TARGET_DP, True),
        ("acc",   "Accuracy (higher=better)", None, False),
    ]
    fig, axes = plt.subplots(n_ds, 3, figsize=(13, 3.8 * n_ds),
                              gridspec_kw={"hspace": 0.55, "wspace": 0.38})
    if n_ds == 1: axes = axes[np.newaxis, :]
    bar_w = 0.65 / n_m

    for di, (ds_key, mr) in enumerate(all_results.items()):
        for ci, (mkey, mlabel, target, _) in enumerate(metrics_cfg):
            ax = axes[di, ci]
            for mi, m in enumerate(active_methods):
                mean, std = _mean_std(mr, m, mkey)
                if np.isnan(mean): continue
                color = METHOD_COLORS.get(m, "#aaa")
                ax.bar(mi, mean, bar_w * 0.88, yerr=std, capsize=3,
                       color=color, alpha=0.85, edgecolor="white", linewidth=0.8)
                seed_vals = [_to_float(v) for v in mr.get(m, {}).get(mkey, [])]
                seed_vals = [v for v in seed_vals if v is not None]
                ax.scatter([mi]*len(seed_vals), seed_vals,
                           color="white", s=18, zorder=5, linewidths=0.8, edgecolors=color)
            if target is not None:
                ax.axhline(target, color="crimson", linestyle="--",
                           linewidth=1.3, alpha=0.8, label=f"target {target}")
                ax.legend(fontsize=6.5, loc="upper right")
            ax.set_xticks(range(n_m))
            ax.set_xticklabels(
                [METHOD_LABELS.get(m, m).replace(" ", "\n") for m in active_methods], fontsize=7)
            ax.set_ylabel(mlabel, fontsize=8)
            ax.set_title(DS_LABELS.get(ds_key, ds_key), fontsize=8.5, pad=4)
            ax.grid(axis="y", alpha=0.25, linewidth=0.7)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

    legend_patches = [mpatches.Patch(color=METHOD_COLORS.get(m, "#aaa"), label=METHOD_LABELS.get(m, m))
                      for m in active_methods]
    fig.legend(handles=legend_patches, loc="upper center", ncol=n_m,
               fontsize=8, bbox_to_anchor=(0.5, 1.01), frameon=False)
    p1 = os.path.join(save_dir, "results_bar.png")
    fig.savefig(p1, dpi=160, bbox_inches="tight")
    if show: plt.show()
    plt.close(fig)
    tqdm.write(f"  [plot] -> {p1}"); saved.append(p1)

    fig2, axes2 = plt.subplots(1, n_ds, figsize=(5.2 * n_ds, 4.5),
                                gridspec_kw={"wspace": 0.35})
    if n_ds == 1: axes2 = [axes2]
    markers = {"vanilla_nn": "o", "zhang_adversarial": "s", "fairlearn_adv": "^", "agad_full": "D"}
    for di, (ds_key, mr) in enumerate(all_results.items()):
        ax = axes2[di]
        for m in active_methods:
            sg_vals = [_to_float(v) for v in mr.get(m, {}).get("sg_eo", [])]
            ac_vals = [_to_float(v) for v in mr.get(m, {}).get("acc",   [])]
            sg_vals = [v for v in sg_vals if v is not None]
            ac_vals = [v for v in ac_vals if v is not None]
            n = min(len(sg_vals), len(ac_vals))
            if n == 0: continue
            ax.scatter(sg_vals[:n], ac_vals[:n],
                       color=METHOD_COLORS.get(m, "#aaa"), marker=markers.get(m, "o"),
                       s=28, alpha=0.45, zorder=3)
            ax.scatter(np.mean(sg_vals[:n]), np.mean(ac_vals[:n]),
                       color=METHOD_COLORS.get(m, "#aaa"), marker=markers.get(m, "o"),
                       s=110, zorder=5, edgecolors="black", linewidths=0.8,
                       label=METHOD_LABELS.get(m, m))
        ax.axvline(TARGET_EO, color="crimson", linestyle="--",
                   linewidth=1.2, alpha=0.75, label=f"EO target={TARGET_EO}")
        ax.set_xlabel("SG-EO (lower = fairer)", fontsize=9)
        ax.set_ylabel("Accuracy", fontsize=9)
        ax.set_title(DS_LABELS.get(ds_key, ds_key), fontsize=10, pad=5)
        ax.legend(fontsize=7, framealpha=0.8)
        ax.grid(alpha=0.25); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    fig2.suptitle("Accuracy-Fairness Pareto Frontier (large=mean, small=per-seed)", fontsize=10, y=1.01)
    p2 = os.path.join(save_dir, "pareto_scatter.png")
    fig2.savefig(p2, dpi=160, bbox_inches="tight")
    if show: plt.show()
    plt.close(fig2)
    tqdm.write(f"  [plot] -> {p2}"); saved.append(p2)

    return saved


def plot_tradeoff_diagnostic(diag_results, save_dir="/kaggle/working", show=True):
    if not HAS_MPL:
        return []
    os.makedirs(save_dir, exist_ok=True)
    n_ds = len(diag_results)
    fig, axes = plt.subplots(1, n_ds, figsize=(5.5 * n_ds, 4.3), gridspec_kw={"wspace": 0.4})
    if n_ds == 1: axes = [axes]
    for di, (ds_key, rows) in enumerate(diag_results.items()):
        ax = axes[di]
        sg_eos = [r["sg_eo"] for r in rows]
        accs   = [r["acc"]   for r in rows]
        dps    = [r["dp"]    for r in rows]
        order  = np.argsort(sg_eos)
        sg_eos_s = np.array(sg_eos)[order]; accs_s = np.array(accs)[order]
        ax.plot(sg_eos_s, accs_s, "-o", color="#3498db", linewidth=1.8, markersize=6, label="Acc vs SG-EO")
        ax2 = ax.twinx()
        dps_s = np.array(dps)[order]
        ax2.plot(sg_eos_s, dps_s, "--s", color="#e67e22", linewidth=1.2, markersize=4, alpha=0.7, label="DP")
        ax2.set_ylabel("WC-DP", fontsize=8, color="#e67e22")
        ax.axvline(TARGET_EO, color="crimson", linestyle=":", linewidth=1.2, alpha=0.7)
        ax.set_xlabel("Validation SG-EO (lower = fairer)", fontsize=9)
        ax.set_ylabel("Validation Accuracy", fontsize=9, color="#3498db")
        ax.set_title(f"{DS_LABELS.get(ds_key, ds_key)}\nFairness-Accuracy Tradeoff Sweep", fontsize=9.5)
        ax.grid(alpha=0.25)
    fig.suptitle("Post-hoc eo_coef/dp_coef sweep: as fairness pressure increases, accuracy moves", fontsize=10, y=1.03)
    p = os.path.join(save_dir, "tradeoff_diagnostic.png")
    fig.savefig(p, dpi=160, bbox_inches="tight")
    if show: plt.show()
    plt.close(fig)
    tqdm.write(f"  [plot] -> {p}")
    return [p]


def save_results(all_results, tuned_params, audit_trails, diag_results, optuna_trial_logs, save_dir="/kaggle/working"):
    def _clean(obj):
        if isinstance(obj, dict):                    return {k: _clean(v) for k, v in obj.items()}
        if isinstance(obj, (list, tuple)):           return [_clean(v) for v in obj]
        if isinstance(obj, float) and np.isnan(obj): return None
        if isinstance(obj, np.floating):             return float(obj)
        if isinstance(obj, np.integer):              return int(obj)
        if isinstance(obj, np.ndarray):              return obj.tolist()
        if isinstance(obj, torch.Tensor):            return obj.tolist()
        return obj
    os.makedirs(save_dir, exist_ok=True)
    path = os.path.join(save_dir, "results_agad_v19.json")
    with open(path, "w") as f:
        json.dump(_clean({
            "results": all_results,
            "tuned_params": tuned_params,
            "audit_trails": audit_trails,
            "tradeoff_diagnostic": diag_results,
            "optuna_trial_logs": optuna_trial_logs,
            "seeds": SEEDS, "version": CACHE_VERSION, "n_seeds": N_SEEDS,
            "methods": METHODS,
            "targets": {"eo": TARGET_EO, "dp": TARGET_DP},
            "config": {
                "MAX_SG_ATTRS": MAX_SG_ATTRS, "TOP_K_INTERSECTIONS": TOP_K_INTERSECTIONS,
                "PLATEAU_PATIENCE": PLATEAU_PATIENCE, "EO_GAP_FIRE_THRESH": EO_GAP_FIRE_THRESH,
                "PARETO_LAMBDA": PARETO_LAMBDA,
                "OPTUNA_N_TRIALS": OPTUNA_N_TRIALS_PER_DS, "OPTUNA_EPOCHS_FRAC": OPTUNA_EPOCHS_FRAC,
            }
        }), f, indent=2)
    tqdm.write(f"  Results -> {path}")
    return path


def main():
    t0 = time.time()
    WALL_CLOCK_BUDGET_MIN = 300

    def _elapsed_min():
        return (time.time() - t0) / 60

    def _phase_check(phase_name):
        e = _elapsed_min()
        tqdm.write(f"  [TIME] {phase_name} done. Elapsed so far: {e:.1f} min"
                   f" / budget {WALL_CLOCK_BUDGET_MIN} min")
        if e > WALL_CLOCK_BUDGET_MIN * 0.9:
            tqdm.write(f"  [TIME WARNING] approaching the {WALL_CLOCK_BUDGET_MIN}-minute budget"
                       f" — remaining phases may need to be cut short or run separately")

    tqdm.write(_banner(
        f"AGAD v19  |  device={DEVICE}  seeds={SEEDS}"
        f"  fairlearn={'yes' if HAS_FAIRLEARN else 'NO'}  optuna={'yes' if HAS_OPTUNA else 'NO'}",
        width=80))
    tqdm.write(f"  Methods: {METHODS}")
    tqdm.write(f"  AGAD loss = task_lw*task_loss + eo_coef*soft_EO + dp_coef*soft_DP + auditor_inter_adv_loss")
    tqdm.write(f"  (no base GRL heads in AGAD; eo_coef and dp_coef are tuned independently)\n")

    all_data = {}
    for ds_key in RUN_DATASETS:
        tqdm.write(f"  Loading {ds_key}...")
        all_data[ds_key] = PREPROCESS_FNS[ds_key]()
    _phase_check("Data loading")

    tqdm.write(_banner("PHASE 1: OPTUNA HYPERPARAMETER TUNING (single pass, accuracy-aware)", width=80))
    optuna_trial_logs = {}
    for ds_key in RUN_DATASETS:
        tuned, trial_log = run_optuna_for_dataset(ds_key, all_data[ds_key])
        TUNED_PARAMS[ds_key] = tuned
        optuna_trial_logs[ds_key] = trial_log
    _phase_check("Phase 1 (Optuna tuning)")

    print_tuned_params_summary(TUNED_PARAMS)

    diag_results = {}
    if RUN_DIAGNOSTIC_SWEEP:
        tqdm.write(_banner("PHASE 2: FAIRNESS-ACCURACY TRADEOFF DIAGNOSTIC SWEEP", width=80))
        for ds_key in RUN_DATASETS:
            diag_results[ds_key] = run_tradeoff_diagnostic(ds_key, all_data[ds_key], TUNED_PARAMS[ds_key])
    _phase_check("Phase 2 (diagnostic sweep)")

    tqdm.write(_banner("PHASE 3: FULL COMPARISON (3 seeds, tuned hyperparameters)", width=80))
    all_results = {}
    for ds_key in RUN_DATASETS:
        tqdm.write(_banner(f"DATASET: {ds_key.upper()}", width=80))
        data = all_data[ds_key]
        cfg  = DATASET_CONFIG[ds_key]
        p    = TUNED_PARAMS[ds_key]
        zp   = ZHANG_PARAMS[ds_key]
        y_te = np.array(data["y_test"])
        sv_p = ensure_binary(np.array(data[f"{cfg['sens_attrs'][0]}_test"]))
        tqdm.write(f"  n_train={len(data['y_train'])}  n_test={len(y_te)}"
                   f"  label_pos={y_te.mean():.3f}  sens_pos={sv_p.mean():.3f}")
        tqdm.write(f"  attrs: {cfg['sens_attrs']}")
        tqdm.write(f"  AGAD tuned: eo_coef={p['eo_coef']:.2f}  dp_coef={p['dp_coef']:.2f}"
                   f"  task_lw={p['task_lw']:.2f}  adv_lr_mult={p['adv_lr_mult']:.2f}")
        ds_t0 = time.time()
        all_results[ds_key], AUDITOR_LOG[ds_key] = _run_dataset(data, ds_key, p, zp)
        tqdm.write(f"  [{ds_key}] wall time: {(time.time()-ds_t0)/60:.1f} min")
        del all_data[ds_key]; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    _phase_check("Phase 3 (3-seed comparison)")

    print_results(all_results)
    print_auditor_report(AUDITOR_LOG)

    plots1 = plot_results(all_results)
    plots2 = plot_tradeoff_diagnostic(diag_results) if diag_results else []
    json_path = save_results(all_results, TUNED_PARAMS, AUDITOR_LOG, diag_results, optuna_trial_logs)

    elapsed = (time.time() - t0) / 60
    tqdm.write(f"\n  Total wall time: {elapsed:.1f} min")
    tqdm.write(f"  Outputs: {json_path}  +  {len(plots1)+len(plots2)} plots")
    return all_results


if __name__ == "__main__":
    main()


  ══ AGAD v19  |  device=cuda  seeds=[42, 43, 44]  fairlearn=yes  optuna=yes ═════
  Methods: ['vanilla_nn', 'zhang_adversarial', 'fairlearn_adv', 'agad_full']
  AGAD loss = task_lw*task_loss + eo_coef*soft_EO + dp_coef*soft_DP + auditor_inter_adv_loss
  (no base GRL heads in AGAD; eo_coef and dp_coef are tuned independently)

  Loading acs_employment...
  [exclude] race_Asian_PI = 0.001 — too small
  Loading adult...
  Loading communities_crime...
  [TIME] Data loading done. Elapsed so far: 0.3 min / budget 300 min

  ══ PHASE 1: OPTUNA HYPERPARAMETER TUNING (single pass, accuracy-aware) ═════════

  ══ OPTUNA TUNING: ACS_EMPLOYMENT ═══════════════════════════════════════════════
  [optuna] acs_employment: subsampled train set to 40000/257595 rows for tuning speed (final 3-seed run uses full data)
    │ pretrain: 8 epochs done

    │ [AUDITOR FIRE #1] epoch=20  val_sg_eo=0.0323 (plateaued 2 evals, best_seen=0.0317)
    │   EO-gap=0.0412  [race_White=1+sex_Female=0] (n=9338, TPR=0.530

    │ [agad]:   0%|          | 0/55 [00:00<?, ?it/s]


    │ [AUDITOR FIRE #1] epoch=16  val_sg_eo=0.0348 (plateaued 2 evals, best_seen=0.0259)
    │   EO-gap=0.0445  [race_Black=1+sex_Female=0] (n=1051, TPR=0.649, FPR=0.254)  vs  [race_Black=1+sex_Female=1] (n=1017, TPR=0.693, FPR=0.266)
    │   EO-gap=0.0373  [race_Black=1+sex_Female=0] (n=1051, TPR=0.649, FPR=0.254)  vs  [race_Other=0+sex_Female=1] (n=10453, TPR=0.686, FPR=0.255)
    │   EO-gap=0.0364  [race_White=1+sex_Female=1] (n=9418, TPR=0.685, FPR=0.254)  vs  [race_Black=1+sex_Female=0] (n=1051, TPR=0.649, FPR=0.254)
    │   EO-gap=0.0359  [race_White=0+sex_Female=1] (n=13638, TPR=0.685, FPR=0.263)  vs  [race_Black=1+sex_Female=0] (n=1051, TPR=0.649, FPR=0.254)
    │   EO-gap=0.0351  [race_Black=1+sex_Female=0] (n=1051, TPR=0.649, FPR=0.254)  vs  [race_Other=1+sex_Female=1] (n=12603, TPR=0.684, FPR=0.262)
    │ [AUDITOR] spawned 5 intersectional adversary heads targeting the pairs above


    │ [AUDITOR FIRE #2] epoch=24  val_sg_eo=0.0285 (plateaued 2 evals, best_seen=0.0259)
   

    │ [agad]:   0%|          | 0/55 [00:00<?, ?it/s]


    │ [AUDITOR FIRE #1] epoch=20  val_sg_eo=0.0228 (plateaued 2 evals, best_seen=0.0186)
    │   EO-gap=0.0386  [race_Black=1+sex_Female=0] (n=1051, TPR=0.661, FPR=0.251)  vs  [race_Black=1+sex_Female=1] (n=1017, TPR=0.699, FPR=0.258)
    │   EO-gap=0.0308  [race_White=0+sex_Female=1] (n=13638, TPR=0.692, FPR=0.259)  vs  [race_Black=1+sex_Female=0] (n=1051, TPR=0.661, FPR=0.251)
    │   EO-gap=0.0303  [race_White=0+sex_Female=1] (n=13638, TPR=0.692, FPR=0.259)  vs  [race_Other=1+sex_Female=0] (n=11998, TPR=0.682, FPR=0.229)
    │   EO-gap=0.0302  [race_Other=1+sex_Female=0] (n=11998, TPR=0.682, FPR=0.229)  vs  [race_Other=1+sex_Female=1] (n=12603, TPR=0.691, FPR=0.259)
    │   EO-gap=0.0302  [race_Black=1+sex_Female=0] (n=1051, TPR=0.661, FPR=0.251)  vs  [race_Other=1+sex_Female=1] (n=12603, TPR=0.691, FPR=0.259)
    │ [AUDITOR] spawned 5 intersectional adversary heads targeting the pairs above


    │ [AUDITOR FIRE #2] epoch=28  val_sg_eo=0.0329 (plateaued 2 evals, best_seen=0.0186)


...


    │ [AUDITOR FIRE #1] epoch=60  val_sg_eo=0.0483 (plateaued 2 evals, best_seen=0.0401)
    │   EO-gap=0.0842  [low_income=1+high_unemployment=0] (n=26, TPR=0.542, FPR=0.500)  vs  [low_income=0+high_unemployment=1] (n=27, TPR=0.626, FPR=0.536)
    │   EO-gap=0.0692  [high_black_pct=1+high_unemployment=0] (n=44, TPR=0.557, FPR=0.528)  vs  [low_income=0+high_unemployment=1] (n=27, TPR=0.626, FPR=0.536)
    │   EO-gap=0.0608  [high_black_pct=0+high_unemployment=1] (n=46, TPR=0.603, FPR=0.522)  vs  [low_income=1+high_unemployment=0] (n=26, TPR=0.542, FPR=0.500)
    │   EO-gap=0.0601  [high_black_pct=0+high_unemployment=0] (n=76, TPR=0.566, FPR=0.489)  vs  [low_income=0+high_unemployment=1] (n=27, TPR=0.626, FPR=0.536)
    │   EO-gap=0.0589  [high_black_pct=1+low_income=0] (n=42, TPR=0.601, FPR=0.534)  vs  [low_income=1+high_unemployment=0] (n=26, TPR=0.542, FPR=0.500)
    │ [AUDITOR] spawned 5 intersectional adversary heads targeting the pairs above


    │ [AUDITOR FIRE #2] epoch=68  va

    │ [agad]:   0%|          | 0/140 [00:00<?, ?it/s]


    │ [AUDITOR FIRE #1] epoch=64  val_sg_eo=0.0411 (plateaued 2 evals, best_seen=0.0387)
    │   EO-gap=0.0520  [high_black_pct=0+low_income=1] (n=43, TPR=0.508, FPR=0.481)  vs  [low_income=0+high_unemployment=1] (n=27, TPR=0.560, FPR=0.487)
    │   EO-gap=0.0500  [high_black_pct=1+low_income=0] (n=42, TPR=0.558, FPR=0.497)  vs  [high_black_pct=0+low_income=1] (n=43, TPR=0.508, FPR=0.481)
    │   EO-gap=0.0453  [high_black_pct=0+low_income=1] (n=43, TPR=0.508, FPR=0.481)  vs  [high_black_pct=1+high_unemployment=1] (n=74, TPR=0.554, FPR=0.481)
    │   EO-gap=0.0400  [high_black_pct=1+low_income=0] (n=42, TPR=0.558, FPR=0.497)  vs  [high_black_pct=0+high_unemployment=1] (n=46, TPR=0.519, FPR=0.483)
    │   EO-gap=0.0398  [high_black_pct=0+low_income=1] (n=43, TPR=0.508, FPR=0.481)  vs  [high_black_pct=1+low_income=1] (n=76, TPR=0.548, FPR=0.483)
    │ [AUDITOR] spawned 5 intersectional adversary heads targeting the pairs above


    │ [AUDITOR FIRE #2] epoch=72  val_sg_eo=0.0454 (platea

    │ [agad]:   0%|          | 0/140 [00:00<?, ?it/s]


    │ [AUDITOR FIRE #1] epoch=12  val_sg_eo=0.0946 (plateaued 2 evals, best_seen=0.0885)
    │   EO-gap=0.1141  [high_black_pct=1+low_income=1] (n=76, TPR=0.607, FPR=0.509)  vs  [low_income=0+high_unemployment=1] (n=27, TPR=0.609, FPR=0.395)
    │   EO-gap=0.1116  [high_black_pct=1+low_income=1] (n=76, TPR=0.607, FPR=0.509)  vs  [high_black_pct=0+high_unemployment=0] (n=76, TPR=0.695, FPR=0.398)
    │   EO-gap=0.1102  [high_black_pct=0+high_unemployment=0] (n=76, TPR=0.695, FPR=0.398)  vs  [high_black_pct=0+high_unemployment=1] (n=46, TPR=0.585, FPR=0.414)
    │   EO-gap=0.1075  [high_black_pct=0+low_income=0] (n=79, TPR=0.654, FPR=0.402)  vs  [high_black_pct=1+low_income=1] (n=76, TPR=0.607, FPR=0.509)
    │   EO-gap=0.1060  [high_black_pct=0+low_income=1] (n=43, TPR=0.594, FPR=0.403)  vs  [high_black_pct=1+low_income=1] (n=76, TPR=0.607, FPR=0.509)
    │ [AUDITOR] spawned 5 intersectional adversary heads targeting the pairs above


    │ [AUDITOR FIRE #2] epoch=20  val_sg_eo=0.0929 